# Part 10 (HFooladi-style): GIN for MDM2 Classification

Colab-ready, self-contained replication of HFooladi/GNNs-For-Chemists (9-dim atom feats, `GIN`, 80/10/10 split seed 42, batch 32, Adam, BCEWithLogitsLoss, best-val-AUC) on the local MDM2 set (`ro5_properties_filtered.csv`).

Run cells top to bottom: install → train → evaluate → screen. Set `EPOCHS = 2` in the training cell for a smoke test.


## 1. Install (Colab)


In [ ]:
!pip install -q torch-geometric rdkit


## 2. Get the repo data (Colab)
If you opened this notebook directly from GitHub, this clones the repo so `ro5_properties_filtered.csv` is found automatically. Set `REPO_URL` to your fork if needed. Skip if the file is already present.


In [ ]:
import os
import subprocess

# --- Setup: make sure the repo (and the MDM2 CSV) exists locally ---
# If you opened this notebook straight from GitHub in Colab, the data file is
# not present yet -- the cell below clones it automatically.
# Edit REPO_URL only if your repo lives somewhere else.
REPO_URL = "https://github.com/arjunpahi/Natural_MDM2_Inhibitor_Discovery_using_ML.git"
REPO_DIR = os.path.basename(REPO_URL).replace(".git", "")


def _find_csv(fname="ro5_properties_filtered.csv", roots=None):
    roots = roots or [os.getcwd(), "/content"]
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dirpath, dirnames, filenames in os.walk(r):
            if ".git" in dirnames:
                dirnames.remove(".git")
            if fname in filenames:
                return os.path.join(dirpath, fname)
            if dirpath.replace(r, "").count(os.sep) >= 3:
                dirnames[:] = []
    return None


if _find_csv() is None and os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR}/ already cloned.")
elif _find_csv() is None:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=False)
print("CSV:", _find_csv() or "NOT FOUND (upload it when asked below)")


In [ ]:
import os
# === Retrofit flags — TASK/both + Delta + filtering + screening (flag-only, reference patterns) ===
TASK = "both"  # "classification" | "regression" | "both" (both = ONE regression run + derived 0/1 at CLASS_THRESHOLD)
USE_DELTA = True
CLASS_THRESHOLD = 7.0  # Active <=> pIC50 >= 7.0 (645-row ro5_properties_filtered.csv)
WEIGHT_DECAY = 1e-4  # unified flag (reuses existing 1e-4 value — now flaggable)
EARLY_STOP = True
PATIENCE = 15
USE_EARLY_STOPPING = EARLY_STOP
EARLY_STOP_PATIENCE = PATIENCE
AUGMENT_N = 2
FILTER_DUP_SMILES = True
FILTER_PIC50_OUTLIER = False
FILTER_INVALID_PIC50 = True
RIDGE_ALPHA = 1.0
PRED_CAP = 10.0
APPLICABILITY_DOMAIN_FILTER = False
STRICT_MEDCHEM = False
FAST_MODE = False
USE_MACCS = False
USE_MORGAN = False
TARGET_COL = "pIC50"
IN_COLAB = os.path.isdir('/content')
print(f'TASK={TASK} | USE_DELTA={USE_DELTA} | CLASS_THRESHOLD={CLASS_THRESHOLD} | WEIGHT_DECAY={WEIGHT_DECAY} | EARLY_STOP={EARLY_STOP}:{PATIENCE} | AUGMENT_N={AUGMENT_N} | DUP={FILTER_DUP_SMILES} OUTLIER={FILTER_PIC50_OUTLIER} INVALID={FILTER_INVALID_PIC50} | RIDGE_ALPHA={RIDGE_ALPHA} | PRED_CAP={PRED_CAP} | AD={APPLICABILITY_DOMAIN_FILTER} | STRICT_MEDCHEM={STRICT_MEDCHEM} | FAST_MODE={FAST_MODE} | MACCS={USE_MACCS} MORGAN={USE_MORGAN} | IN_COLAB={IN_COLAB}')


## 2. Imports + paths
Works from the repo root (`Part_10/` auto-detected) or from inside `Part_10/` itself — e.g. after cloning your repo in Colab and `%cd`-ing into it.


In [ ]:
"""Part 10 (HFooladi-exact): GIN for MDM2 inhibitor classification.

Faithful replication of HFooladi/GNNs-For-Chemists notebook 06
(`06_GNN_GIN.ipynb`) applied to the local MDM2 dataset:

* Featurization — byte-identical to notebook 04 / ``notebooks/utils/featurization.py``
  (also imported, not redefined, from notebook 05 onward in HFooladi):
  9-dim integer atom features + 3-dim bond features, ``mol_to_graph(smiles)``.
* Model — ``MLP`` + ``GIN`` identical to notebook 06: ``node_encoder`` Linear,
  ``num_layers`` × ``GINConv(MLP, train_eps=True)`` + BatchNorm + ReLU + Dropout,
  ``global_mean_pool``, final prediction ``MLP``; plus ``get_embeddings``.
* Training — ``train_and_evaluate`` identical to notebook 06 (BBBP setup):
  ``BCEWithLogitsLoss``, 80/10/10 random split (seed 42), ``DataLoader``
  batch 32, Adam (lr 1e-3, weight_decay 1e-4), 100 epochs, best-val-AUC
  checkpointing, ROC-AUC reporting.

Dataset: ``Part_10/ro5_properties_filtered.csv`` (645 MDM2 compounds,
``bioactivity_class`` -> y in {0, 1}).

Usage:
    python train_gin_hfooladi.py
    python train_gin_hfooladi.py --epochs 100 --hidden 64 --layers 3
    python train_gin_hfooladi.py --screen input.csv --screen-out preds.csv
"""
import os

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool

BASE = os.path.dirname(_find_csv())
DATA_CSV = _find_csv()
if DATA_CSV is None:
    # Last resort: ask for the file.
    BASE = os.getcwd()
    DATA_CSV = os.path.join(BASE, "ro5_properties_filtered.csv")
    try:
        from google.colab import files as _cf
        print(f"{DATA_CSV} not found. Please upload ro5_properties_filtered.csv:")
        _up = _cf.upload()
        _name = list(_up.keys())[0]
        if os.path.basename(_name) == os.path.basename(DATA_CSV):
            with open(DATA_CSV, "wb") as _f:
                _f.write(_up[_name])
        else:
            DATA_CSV = _name
        print(f"Using {DATA_CSV}")
    except ImportError:
        raise FileNotFoundError(
            f"{DATA_CSV} not found. Put ro5_properties_filtered.csv in {BASE} "
            "next to the notebook (or %cd to the repo root).")
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ---------------------------------------------------------------------------
# Featurization — identical to HFooladi notebooks 04/06
# ---------------------------------------------------------------------------
def atom_features(atom):
    """Extract a feature vector for an RDKit atom (shape (9,), dtype long)."""
    return torch.tensor([
        atom.GetAtomicNum(),
        int(atom.GetChiralTag()),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetTotalNumHs(),
        atom.GetNumRadicalElectrons(),
        int(atom.GetHybridization()),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing()),
    ], dtype=torch.long)


def bond_features(bond):
    """Extract a feature vector for an RDKit bond (shape (3,), dtype long)."""
    return torch.tensor([
        int(bond.GetBondTypeAsDouble()),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
    ], dtype=torch.long)


def mol_to_graph(smiles):
    """Convert a SMILES into a PyTorch Geometric graph (undirected, 2 directed
    edges per bond). Returns Data(x [N,9], edge_index [2,E], edge_attr [E,3])."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")
    x = torch.stack([atom_features(atom) for atom in mol.GetAtoms()], dim=0)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append((i, j))
        edge_index.append((j, i))
        edge_attr.append(bond_features(bond))
        edge_attr.append(bond_features(bond))
    if edge_index:  # guard for single-atom molecules (no bonds); HFooladi's
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()  # tutorial set has none
        edge_attr = torch.stack(edge_attr, dim=0)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = None
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, smiles=smiles)


def mol_to_graph_labeled(smiles, y):
    """mol_to_graph + float label ``y`` of shape [1] for BCEWithLogitsLoss."""
    try:
        d = mol_to_graph(smiles)
    except ValueError:
        return None
    d.y = torch.tensor([[float(y)]], dtype=torch.float)
    return d




## 3. Model — `GIN` (identical to HFooladi)


In [ ]:
# ---------------------------------------------------------------------------
# Model — identical to HFooladi notebook 06 (MLP + GIN)
# ---------------------------------------------------------------------------
class MLP(nn.Module):
    """MLP used both in GINConv layers and for final predictions."""

    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2):
        super(MLP, self).__init__()
        self.lins = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        if num_layers == 1:
            self.lins.append(nn.Linear(in_channels, out_channels))
        else:
            self.lins.append(nn.Linear(in_channels, hidden_channels))
            self.batch_norms.append(nn.BatchNorm1d(hidden_channels))
            for _ in range(num_layers - 2):
                self.lins.append(nn.Linear(hidden_channels, hidden_channels))
                self.batch_norms.append(nn.BatchNorm1d(hidden_channels))
            self.lins.append(nn.Linear(hidden_channels, out_channels))

    def forward(self, x):
        for i, lin in enumerate(self.lins[:-1]):
            x = lin(x)
            x = self.batch_norms[i](x)
            x = F.relu(x)
        x = self.lins[-1](x)
        return x


class GIN(nn.Module):
    """Graph Isomorphism Network: node_encoder + GINConv layers + mean pool + MLP."""

    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3,
                 dropout=0.5, epsilon=0):
        super(GIN, self).__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.node_encoder = nn.Linear(in_channels, hidden_channels)
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            mlp = MLP(hidden_channels, hidden_channels, hidden_channels)
            self.convs.append(GINConv(mlp, train_eps=True, eps=epsilon))
        self.batch_norms = nn.ModuleList(
            [nn.BatchNorm1d(hidden_channels) for _ in range(num_layers)])
        self.mlp = MLP(hidden_channels, hidden_channels, out_channels)

    def forward(self, x, edge_index, batch):
        x = self.node_encoder(x.float())
        xs = []
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.batch_norms[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            xs.append(x)
        x = global_mean_pool(x, batch)
        x = self.mlp(x)
        return x

    def get_embeddings(self, x, edge_index, batch):
        """Node embeddings from all layers (no dropout; for visualisation)."""
        x = self.node_encoder(x.float())
        xs = []
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.batch_norms[i](x)
            x = F.relu(x)
            xs.append(x)
        return xs




## 4. Training loop + data helpers (identical to HFooladi notebook 06)


In [ ]:
# ---------------------------------------------------------------------------
# Training — identical to HFooladi notebook 06 train_and_evaluate
# ---------------------------------------------------------------------------
def train_and_evaluate(model, optimizer, train_loader, val_loader, test_loader,
                       dev, epochs=100):
    """Train and evaluate a model for molecular property prediction."""
    train_losses, val_losses, val_aucs = [], [], []
    best_val_auc, best_model = 0, None
    _no_improve = 0  # retrofit: early stopping counter (val AUC, patience 15, restore best)
    _PATIENCE = int(globals().get("PATIENCE", globals().get("EARLY_STOP_PATIENCE", 15)))
    _EARLY_STOP = bool(globals().get("EARLY_STOP", globals().get("USE_EARLY_STOPPING", True)))
    criterion = nn.BCEWithLogitsLoss()
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_samples = 0, 0
        for data in train_loader:
            optimizer.zero_grad()
            data = data.to(dev)
            out = model(data.x.float(), data.edge_index, data.batch)
            if out.size(0) != data.y.size(0):
                m = min(out.size(0), data.y.size(0))
                out, data.y = out[:m], data.y[:m]
            loss = criterion(out, data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
            total_samples += data.num_graphs
        avg_loss = total_loss / total_samples
        train_losses.append(avg_loss)

        model.eval()
        val_loss, val_samples, y_true, y_pred = 0, 0, [], []
        with torch.no_grad():
            for data in val_loader:
                data = data.to(dev)
                out = model(data.x.float(), data.edge_index, data.batch)
                if out.size(0) != data.y.size(0):
                    m = min(out.size(0), data.y.size(0))
                    out, data.y = out[:m], data.y[:m]
                loss = criterion(out, data.y)
                val_loss += loss.item() * data.num_graphs
                val_samples += data.num_graphs
                y_true.append(data.y.cpu().numpy())
                y_pred.append(torch.sigmoid(out).cpu().numpy())
        avg_val_loss = val_loss / val_samples
        val_losses.append(avg_val_loss)
        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)
        val_auc = roc_auc_score(y_true, y_pred)
        val_aucs.append(val_auc)
        if val_auc > best_val_auc + 1e-4:
            best_val_auc = val_auc
            best_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            _no_improve = 0
        else:
            _no_improve += 1
        if _EARLY_STOP and _no_improve >= _PATIENCE:
            print(f"Early stopping at epoch {epoch} (best Val AUC {best_val_auc:.4f}, patience {_PATIENCE})", flush=True)
            break
        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d}: Train Loss: {avg_loss:.4f}, "
                  f"Val Loss: {avg_val_loss:.4f}, Val AUC: {val_auc:.4f}", flush=True)

    if best_model is not None:
        model.load_state_dict(best_model)
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(dev)
            out = model(data.x.float(), data.edge_index, data.batch)
            if out.size(0) != data.y.size(0):
                m = min(out.size(0), data.y.size(0))
                out, data.y = out[:m], data.y[:m]
            y_true.append(data.y.cpu().numpy())
            y_pred.append(torch.sigmoid(out).cpu().numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    test_auc = roc_auc_score(y_true, y_pred)
    return {"train_losses": train_losses, "val_losses": val_losses,
            "val_aucs": val_aucs, "best_val_auc": best_val_auc,
            "test_auc": test_auc, "y_true": y_true, "y_pred": y_pred}


def load_mdm2_graphs(csv_path=DATA_CSV):
    df = pd.read_csv(csv_path)
    df["y"] = (df["bioactivity_class"] == "Active").astype(int)
    graphs = [g for g in (mol_to_graph_labeled(s, y)
                          for s, y in zip(df["canonical_smiles"], df["y"]))
              if g is not None]
    print(f"Converted {len(graphs)}/{len(df)} molecules to graphs "
          f"(9-dim atom feats: {graphs[0].x.shape})")
    return graphs, df


def random_split(graphs, seed=SEED):
    torch.manual_seed(seed)
    idx = torch.randperm(len(graphs))
    n = len(graphs)
    tr, va = idx[:int(0.8 * n)], idx[int(0.8 * n):int(0.9 * n)]
    te = idx[int(0.9 * n):]
    return [graphs[i] for i in tr], [graphs[i] for i in va], [graphs[i] for i in te]

# --- Retrofit: Delta-ML regression helpers (reference colabtestregressiom_fast patterns, adapted to 9-dim HFooladi graphs) ---
# Ridge train-only + StandardScaler + MSELoss + R2/RMSE/MAE; Delta head out_dim=1 (same arch); PRED_CAP clip; AUGMENT_N wiring
try:
    from sklearn.linear_model import Ridge as _Ridge
    from sklearn.preprocessing import StandardScaler as _StandardScaler
    from sklearn.metrics import mean_squared_error as _mse_fn, mean_absolute_error as _mae_fn, r2_score as _r2_fn
    _HAVE_REG = True
except Exception:
    _HAVE_REG = False

def rdkit_desc(smi):
    """6-dim RDKit descriptors for Ridge baseline (train-only)."""
    try:
        from rdkit.Chem import Descriptors as _D
        from rdkit import Chem as _C
        mol = _C.MolFromSmiles(str(smi))
        if mol is None:
            return [0.0]*6
        return [float(_D.MolWt(mol)), float(_D.MolLogP(mol)), float(_D.TPSA(mol)),
                float(_D.NumHAcceptors(mol)), float(_D.NumHDonors(mol)), float(_D.NumRotatableBonds(mol))]
    except Exception:
        return [0.0]*6

def enumerated_smiles(smi, n=None):
    """Train-only SMILES enumeration (augmentation x AUGMENT_N)."""
    try:
        from rdkit import Chem as _C
    except Exception:
        return [smi]
    if n is None:
        try:
            n = int(globals().get("AUGMENT_N", 2))
        except Exception:
            n = 2
    try:
        mol = _C.MolFromSmiles(str(smi))
        if mol is None:
            return [smi]
        outs = []
        for seed in range(max(1, n) * 5):
            try:
                s = _C.MolToSmiles(mol, doRandom=True, canonical=False, rootedAtAtom=seed % max(1, mol.GetNumAtoms()))
                if s not in outs:
                    outs.append(s)
                if len(outs) >= n:
                    break
            except Exception:
                continue
        if not outs:
            try:
                outs = [_C.MolToSmiles(mol)]
            except Exception:
                outs = [smi]
        return outs
    except Exception:
        return [smi]

def train_and_evaluate_reg(model, optimizer, train_loader, val_loader, test_loader, dev, epochs=100, scaler=None, baseline_val=None, baseline_test=None):
    """Regression trainer (Delta-ML, MSELoss, val-R2 early stopping patience 15, restore best)."""
    import numpy as _np
    import torch as _torch
    import torch.nn as _nn
    try:
        from sklearn.metrics import r2_score as _r2l
    except Exception:
        _r2l = None
    train_losses, val_losses, val_r2s, val_aucs = [], [], [], []
    best_val_r2, best_val_auc, best_model = -1e9, 0.0, None
    _no_improve = 0
    try:
        _PATIENCE = int(globals().get("PATIENCE", globals().get("EARLY_STOP_PATIENCE", 15)))
    except Exception:
        _PATIENCE = 15
    try:
        _EARLY_STOP = bool(globals().get("EARLY_STOP", globals().get("USE_EARLY_STOPPING", True)))
    except Exception:
        _EARLY_STOP = True
    try:
        _CAP = float(globals().get("PRED_CAP", 10.0))
    except Exception:
        _CAP = 10.0
    try:
        _THR = float(globals().get("CLASS_THRESHOLD", 7.0))
    except Exception:
        _THR = 7.0
    criterion = _nn.MSELoss()
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        for data in train_loader:
            optimizer.zero_grad()
            data = data.to(dev)
            out = model(data.x.float(), data.edge_index, data.batch)
            if out.size(0) != data.y.size(0):
                m = min(out.size(0), data.y.size(0))
                out, data.y = out[:m], data.y[:m]
            y = data.y.view(-1).to(out.dtype)
            loss = criterion(out.view(-1), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * data.num_graphs
            total_n += data.num_graphs
        avg_loss = total_loss / max(1, total_n)
        train_losses.append(avg_loss)
        # validate
        model.eval()
        _yt, _yp, _vloss, _vn = [], [], 0.0, 0
        with _torch.no_grad():
            for data in val_loader:
                data = data.to(dev)
                out = model(data.x.float(), data.edge_index, data.batch)
                if out.size(0) != data.y.size(0):
                    m = min(out.size(0), data.y.size(0))
                    out, data.y = out[:m], data.y[:m]
                y = data.y.view(-1).to(out.dtype)
                _vloss += criterion(out.view(-1), y).item() * data.num_graphs
                _vn += data.num_graphs
                _yt.append(data.y.cpu().numpy().ravel())
                _yp.append(out.cpu().numpy().ravel())
        import numpy as _np2
        _yt = _np2.concatenate(_yt) if len(_yt) else _np2.array([])
        _yp = _np2.concatenate(_yp) if len(_yp) else _np2.array([])
        avg_val_loss = _vloss / max(1, _vn)
        val_losses.append(avg_val_loss)
        # to pIC50 for R2/AUC (inverse + baseline if Delta)
        try:
            if scaler is not None:
                yt_o = scaler.inverse_transform(_yt.reshape(-1, 1)).ravel()
                yp_o = scaler.inverse_transform(_yp.reshape(-1, 1)).ravel()
                if bool(globals().get("USE_DELTA", True)) and baseline_val is not None:
                    yt_o = yt_o + _np2.asarray(baseline_val).ravel()[:len(yt_o)]
                    yp_o = yp_o + _np2.asarray(baseline_val).ravel()[:len(yp_o)]
                yp_o = _np2.clip(yp_o, None, _CAP)
            else:
                yt_o, yp_o = _yt, _np2.clip(_yp, None, _CAP)
        except Exception:
            yt_o, yp_o = _yt, _yp
        try:
            from sklearn.metrics import r2_score as _rr, roc_auc_score as _ar
            val_r2 = float(_rr(yt_o, yp_o)) if len(yt_o) > 1 else 0.0
            try:
                yb_true = (yt_o >= _THR).astype(int)
                val_auc = float(_ar(yb_true, yp_o)) if len(set(yb_true.tolist())) > 1 else 0.5
            except Exception:
                val_auc = 0.5
        except Exception:
            val_r2, val_auc = 0.0, 0.5
        val_r2s.append(val_r2)
        val_aucs.append(val_auc)
        if val_r2 > best_val_r2 + 1e-4:
            best_val_r2 = val_r2
            best_val_auc = val_auc
            best_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            _no_improve = 0
        else:
            _no_improve += 1
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d}: Train Loss: {avg_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val R2: {val_r2:.4f}, Val AUC(derived): {val_auc:.4f}", flush=True)
        if _EARLY_STOP and _no_improve >= _PATIENCE:
            print(f"Early stopping (reg) at epoch {epoch} (best Val R2 {best_val_r2:.4f}, patience {_PATIENCE})", flush=True)
            break
    if best_model is not None:
        model.load_state_dict(best_model)
    # test
    model.eval()
    _tte, _tpe = [], []
    with _torch.no_grad():
        for data in test_loader:
            data = data.to(dev)
            out = model(data.x.float(), data.edge_index, data.batch)
            if out.size(0) != data.y.size(0):
                m = min(out.size(0), data.y.size(0))
                out, data.y = out[:m], data.y[:m]
            _tte.append(data.y.cpu().numpy().ravel())
            _tpe.append(out.cpu().numpy().ravel())
    import numpy as _np3
    _tte = _np3.concatenate(_tte) if len(_tte) else _np3.array([])
    _tpe = _np3.concatenate(_tpe) if len(_tpe) else _np3.array([])
    try:
        if scaler is not None:
            tte_o = scaler.inverse_transform(_tte.reshape(-1, 1)).ravel()
            tpe_o = scaler.inverse_transform(_tpe.reshape(-1, 1)).ravel()
            if bool(globals().get("USE_DELTA", True)) and baseline_test is not None:
                tte_o = tte_o + _np3.asarray(baseline_test).ravel()[:len(tte_o)]
                tpe_o = tpe_o + _np3.asarray(baseline_test).ravel()[:len(tpe_o)]
            tpe_o = _np3.clip(tpe_o, None, _CAP)
        else:
            tte_o, tpe_o = _tte, _tpe
    except Exception:
        tte_o, tpe_o = _tte, _tpe
    try:
        from sklearn.metrics import mean_squared_error as _mse2, mean_absolute_error as _mae2, r2_score as _r22, roc_auc_score as _au2
        test_rmse = float(_mse2(tte_o, tpe_o) ** 0.5)
        test_mae = float(_mae2(tte_o, tpe_o))
        test_r2 = float(_r22(tte_o, tpe_o)) if len(tte_o) > 1 else 0.0
        try:
            yb_t = (tte_o >= _THR).astype(int)
            test_auc = float(_au2(yb_t, tpe_o)) if len(set(yb_t.tolist())) > 1 else 0.5
        except Exception:
            test_auc = 0.5
    except Exception:
        test_rmse, test_mae, test_r2, test_auc = 0.0, 0.0, 0.0, 0.5
    # derived classification probs for ROC/confusion compat (sigmoid of distance to threshold)
    try:
        import numpy as _np4
        y_true_cls = (tte_o >= _THR).astype(int).reshape(-1, 1)
        y_prob_derived = (1.0 / (1.0 + _np4.exp(-(tpe_o - _THR)))).reshape(-1, 1)
    except Exception:
        y_true_cls, y_prob_derived = tte_o.reshape(-1, 1), tpe_o.reshape(-1, 1)
    return {"train_losses": train_losses, "val_losses": val_losses, "val_r2s": val_r2s, "val_aucs": val_aucs,
            "best_val_r2": best_val_r2, "best_val_auc": best_val_auc,
            "test_rmse": test_rmse, "test_mae": test_mae, "test_r2": test_r2, "test_auc": test_auc,
            "y_true_pic50": tte_o.reshape(-1, 1), "y_pred_pic50": tpe_o.reshape(-1, 1),
            "y_true": y_true_cls, "y_pred": y_prob_derived}



## 5. Screening helper (same `mol_to_graph → sigmoid(model)` pattern)


In [ ]:
# ---------------------------------------------------------------------------
# Screening — same mol_to_graph -> sigmoid(model) pattern as training
# ---------------------------------------------------------------------------
@torch.no_grad()
def screen_smiles(model, smiles_list, dev, batch_size=256):
    """Return P(active) for each valid SMILES (NaN for invalid)."""
    model.eval()
    probs = np.full(len(smiles_list), np.nan)
    graphs, keep = [], []
    for i, s in enumerate(smiles_list):
        try:
            graphs.append(mol_to_graph(s))
            keep.append(i)
        except ValueError:
            continue
    for s in range(0, len(graphs), batch_size):
        chunk = graphs[s:s + batch_size]
        loader = DataLoader(chunk, batch_size=len(chunk), shuffle=False)
        b = next(iter(loader)).to(dev)
        p = torch.sigmoid(model(b.x.float(), b.edge_index, b.batch))[:, 0]
        probs[np.array(keep[s:s + batch_size])] = p.cpu().numpy()
    return probs


PROB_COL = "gin_hfooladi_prob"

SMILES_CANDIDATES = ["canonical_smiles", "smiles", "SMILES", "isomeric_smiles",
                     "structure", "clean_smiles"]
ID_CANDIDATES = ["identifier", "coconut_id", "coconutID", "id", "name",
                 "title", "compound_id"]


def resolve_columns(columns, smiles_col="canonical_smiles"):
    """Pick the SMILES + identifier columns, tolerating COCONUT-style names."""
    scol = smiles_col if smiles_col in columns else next(
        (c for c in SMILES_CANDIDATES if c in columns), columns[0])
    idcol = next((c for c in ID_CANDIDATES if c in columns and c != scol), None)
    return scol, idcol


def screen_file(model, input_csv, output_csv, dev, smiles_col="canonical_smiles",
                chunksize=20000, batch_size=256):
    """Memory-safe screening of large libraries (e.g. COCONUT, ~10^5-10^6 rows).

    Reads the input CSV in chunks, featurizes+predicts per chunk, keeps only
    identifier/SMILES/probability, then writes one ranked CSV (NaN last).
    """
    cols = pd.read_csv(input_csv, nrows=0).columns.tolist()
    scol, idcol = resolve_columns(cols, smiles_col)
    keep = ([idcol] if idcol else []) + [scol]
    print(f"SMILES column: {scol} | ID column: {idcol}")
    parts, total, valid = [], 0, 0
    reader = pd.read_csv(input_csv, chunksize=chunksize, usecols=keep)
    for chunk in reader:
        smiles = chunk[scol].astype(str).tolist()
        probs = screen_smiles(model, smiles, dev, batch_size=batch_size)
        out = pd.DataFrame({scol: smiles, PROB_COL: probs})
        if idcol:
            out.insert(0, idcol, chunk[idcol].values)
        parts.append(out)
        total += len(chunk)
        valid += int(np.isfinite(probs).sum())
        print(f"  screened {total} ({valid} valid) ...", flush=True)
    full = pd.concat(parts, ignore_index=True)
    full.sort_values(PROB_COL, ascending=False).to_csv(output_csv, index=False)
    print(f"Screened {total} compounds ({valid} valid) -> {output_csv}")




## 6. Train on MDM2


In [ ]:
# Hyperparameters (HFooladi notebook 06 defaults for classification) — retrofit: WD unified to WEIGHT_DECAY flag, TASK branch
EPOCHS = 100
HIDDEN = 64
LAYERS = 3
DROPOUT = 0.5
LR = 1e-3
WD = WEIGHT_DECAY  # retrofit: unified flag (was 1e-4, same value now flaggable)
TASK_EFF = globals().get("TASK", "both")
CLASS_THR = float(globals().get("CLASS_THRESHOLD", 7.0))
_PRED_CAP = float(globals().get("PRED_CAP", 10.0))
print(f"TASK={TASK_EFF} | CLASS_THRESHOLD={CLASS_THR} | WEIGHT_DECAY={WEIGHT_DECAY} | PRED_CAP={_PRED_CAP}")

if TASK_EFF == "classification":
    # --- existing classification path (hidden dims/LRs unchanged) ---
    model = GIN(9, HIDDEN, out_channels=1, num_layers=LAYERS,
    dropout=DROPOUT, epsilon=0).to(device)
    print(model)
    print(f"Params: {sum(p.numel() for p in model.parameters()):,} | device: {device}")
    graphs, _ = load_mdm2_graphs()
    tr_graphs, va_graphs, te_graphs = random_split(graphs)
    print(f"Train: {len(tr_graphs)}, Validation: {len(va_graphs)}, Test: {len(te_graphs)}")
    tr_loader = DataLoader(tr_graphs, batch_size=32, shuffle=True)
    va_loader = DataLoader(va_graphs, batch_size=32, shuffle=False)
    te_loader = DataLoader(te_graphs, batch_size=32, shuffle=False)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    res = train_and_evaluate(model, opt, tr_loader, va_loader, te_loader, device, EPOCHS)
    print(f"Best Val AUC: {res['best_val_auc']:.4f} | Test AUC: {res['test_auc']:.4f}")
else:
    # --- regression / both: ONE Delta-ML regression run + derived 0/1 at CLASS_THRESHOLD ---
    # Ridge train-only + StandardScaler + MSELoss + R2/RMSE/MAE; Delta head out_dim=1 (same arch); AUGMENT_N; filtering flags; PRED_CAP clip
    from sklearn.linear_model import Ridge as _Ridge2
    from sklearn.preprocessing import StandardScaler as _SC2
    from sklearn.metrics import mean_squared_error as _mse2, mean_absolute_error as _mae2, r2_score as _r22, roc_auc_score as _auc2, accuracy_score as _acc2, f1_score as _f12
    import numpy as _np_reg
    # load + filter (flags)
    _df = pd.read_csv(DATA_CSV)
    print(f"Raw rows: {len(_df)}")
    _tcol = globals().get("TARGET_COL", "pIC50")
    if bool(globals().get("FILTER_INVALID_PIC50", True)):
        _before = len(_df)
        _df = _df[pd.notna(_df[_tcol])].copy()
        print(f"Invalid pIC50 dropped: {_before - len(_df)}")
    if bool(globals().get("FILTER_DUP_SMILES", True)) and "canonical_smiles" in _df.columns:
        _before = len(_df)
        _df = _df.sort_values(_tcol, ascending=False).drop_duplicates("canonical_smiles", keep="first")
        print(f"Dup SMILES removed: {_before - len(_df)} -> {len(_df)}")
    if bool(globals().get("FILTER_PIC50_OUTLIER", False)):
        _q1, _q3 = _df[_tcol].quantile([0.25, 0.75])
        _iqr = _q3 - _q1
        _lo, _hi = _q1 - 1.5 * _iqr, _q3 + 1.5 * _iqr
        _before = len(_df)
        _df = _df[(_df[_tcol] >= _lo) & (_df[_tcol] <= _hi)].copy()
        print(f"IQR outlier filter [{_lo:.2f},{_hi:.2f}] removed {_before - len(_df)}")
    # descriptors + Ridge baseline candidates
    _desc_mat = _np_reg.array([rdkit_desc(s) for s in _df["canonical_smiles"].tolist()], dtype=float)
    _raw = _df[_tcol].values.astype(float)
    print(f"Descriptors: {_desc_mat.shape} | pIC50 range [{_raw.min():.2f}, {_raw.max():.2f}]")
    # graphs with continuous y (raw for now; scaled after split)
    _graphs_all, _valid = [], []
    for _idx, _row in _df.reset_index(drop=True).iterrows():
        _g = mol_to_graph_labeled(str(_row["canonical_smiles"]), float(_row[_tcol]))
        if _g is not None:
            _graphs_all.append(_g)
            _valid.append(_idx)
    _df = _df.reset_index(drop=True).loc[_valid].reset_index(drop=True)
    _desc_mat = _desc_mat[_valid]
    _raw = _raw[_valid]
    print(f"Converted {len(_graphs_all)}/{len(_valid)} molecules to graphs")
    graphs = _graphs_all
    # 80/10/10 split (seed 42, same as classification)
    torch.manual_seed(SEED)
    _perm = torch.randperm(len(graphs))
    _n = len(graphs)
    _tri, _vai = _perm[:int(0.8*_n)].tolist(), _perm[int(0.8*_n):int(0.9*_n)].tolist()
    _tei = _perm[int(0.9*_n):].tolist()
    tr_graphs = [graphs[i] for i in _tri]
    va_graphs = [graphs[i] for i in _vai]
    te_graphs = [graphs[i] for i in _tei]
    print(f"Train: {len(tr_graphs)}, Validation: {len(va_graphs)}, Test: {len(te_graphs)}")
    # Ridge train-only + scaler (Delta-aware, no leakage)
    _tr_desc, _va_desc, _te_desc = _desc_mat[_tri], _desc_mat[_vai], _desc_mat[_tei]
    _tr_raw, _va_raw, _te_raw = _raw[_tri], _raw[_vai], _raw[_tei]
    baseline_model = _Ridge2(alpha=float(globals().get("RIDGE_ALPHA", 1.0)))
    baseline_model.fit(_tr_desc, _tr_raw)
    _tr_base, _va_base, _te_base = baseline_model.predict(_tr_desc), baseline_model.predict(_va_desc), baseline_model.predict(_te_desc)
    print(f"Ridge baseline R2 train: {baseline_model.score(_tr_desc, _tr_raw):.3f} val: {baseline_model.score(_va_desc, _va_raw):.3f} test: {baseline_model.score(_te_desc, _te_raw):.3f}")
    _USE_DELTA = bool(globals().get("USE_DELTA", True))
    if _USE_DELTA:
        _tr_y, _va_y, _te_y = _tr_raw - _tr_base, _va_raw - _va_base, _te_raw - _te_base
    else:
        _tr_y, _va_y, _te_y = _tr_raw, _va_raw, _te_raw
    target_scaler = _SC2().fit(_tr_y.reshape(-1, 1))
    _tr_s = target_scaler.transform(_tr_y.reshape(-1, 1)).ravel()
    _va_s = target_scaler.transform(_va_y.reshape(-1, 1)).ravel()
    _te_s = target_scaler.transform(_te_y.reshape(-1, 1)).ravel()
    for _gi, _yv in zip(_tri, _tr_s):
        graphs[_gi].y = torch.tensor([[float(_yv)]], dtype=torch.float)
    for _gi, _yv in zip(_vai, _va_s):
        graphs[_gi].y = torch.tensor([[float(_yv)]], dtype=torch.float)
    for _gi, _yv in zip(_tei, _te_s):
        graphs[_gi].y = torch.tensor([[float(_yv)]], dtype=torch.float)
    # refresh split lists (y updated)
    tr_graphs = [graphs[i] for i in _tri]
    va_graphs = [graphs[i] for i in _vai]
    te_graphs = [graphs[i] for i in _tei]
    # augmentation train-only x AUGMENT_N (enumerated SMILES, scaled labels)
    _AUG = int(globals().get("AUGMENT_N", 2))
    _tr_smis = _df.iloc[_tri]["canonical_smiles"].tolist() if "canonical_smiles" in _df.columns else [g.smiles for g in tr_graphs]
    _aug_pairs = [(s, float(y)) for smi, y in zip(_tr_smis, _tr_s) for s in enumerated_smiles(smi, _AUG)]
    _aug_graphs = []
    for _s, _y in _aug_pairs:
        try:
            _ag = mol_to_graph(str(_s))
            if _ag is not None:
                _ag.y = torch.tensor([[float(_y)]], dtype=torch.float)
                _aug_graphs.append(_ag)
        except Exception:
            continue
    if len(_aug_graphs):
        tr_loader = DataLoader(_aug_graphs, batch_size=32, shuffle=True)
        print(f"Augmented train pool: {len(_aug_graphs)} graphs (AUGMENT_N={_AUG})")
    else:
        tr_loader = DataLoader(tr_graphs, batch_size=32, shuffle=True)
    va_loader = DataLoader(va_graphs, batch_size=32, shuffle=False)
    te_loader = DataLoader(te_graphs, batch_size=32, shuffle=False)
    # Delta head out_dim=1 (same arch, regression)
    model = GIN(9, HIDDEN, out_channels=1, num_layers=LAYERS,
    dropout=DROPOUT, epsilon=0).to(device)
    print(model)
    print(f"Params: {sum(p.numel() for p in model.parameters()):,} | device: {device} | Delta={_USE_DELTA} Ridge_alpha={float(globals().get('RIDGE_ALPHA', 1.0))}")
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    res = train_and_evaluate_reg(model, opt, tr_loader, va_loader, te_loader, device, EPOCHS, scaler=target_scaler, baseline_val=_va_base, baseline_test=_te_base)
    # store baselines for screening reuse
    globals()["baseline_val"] = _va_base
    globals()["baseline_test"] = _te_base
    print(f"Best Val R2: {res['best_val_r2']:.4f} (AUC derived {res['best_val_auc']:.4f}) | Test R2: {res['test_r2']:.4f} RMSE: {res['test_rmse']:.4f} MAE: {res['test_mae']:.4f} AUC(derived): {res['test_auc']:.4f}")
    # derived 0/1 at CLASS_THRESHOLD already in res[y_true/y_pred] for ROC/confusion compat
    _yt_c = res["y_true"].ravel().astype(int)
    _yp_prob = res["y_pred"].ravel()
    _yb = (_yp_prob > 0.5).astype(int)
    # also report pIC50-space derived at CLASS_THRESHOLD
    _yt_p = res["y_true_pic50"].ravel()
    _yp_p = res["y_pred_pic50"].ravel()
    _yb_p = (_yp_p >= CLASS_THR).astype(int)
    print(f"Derived cls @ {CLASS_THR} (prob>0.5): Acc {_acc2(_yt_c, _yb):.4f} F1 {_f12(_yt_c, _yb, zero_division=0):.4f} AUC {_auc2(_yt_c, _yp_prob):.4f}")
    print(f"Derived cls @ {CLASS_THR} (pIC50>={CLASS_THR}): Acc {_acc2((_yt_p>=CLASS_THR).astype(int), _yb_p):.4f}")


## 7. Evaluate + save


In [ ]:
# retrofit: TASK-aware evaluate (both modes) — regression reports R2/RMSE/MAE + derived cls; classification unchanged
_TASK_EVAL = globals().get("TASK", "classification")
_CLASS_THR_EVAL = float(globals().get("CLASS_THRESHOLD", 7.0))
_PRED_CAP_EVAL = float(globals().get("PRED_CAP", 10.0))
_MODE_EVAL = f"TASK={_TASK_EVAL} cap={_PRED_CAP_EVAL}"
if _TASK_EVAL in ("regression", "both") and "y_true_pic50" in res:
    import numpy as _np_ev
    from sklearn.metrics import accuracy_score as _acc_ev, precision_score as _pr_ev, recall_score as _rc_ev, f1_score as _f1_ev, roc_auc_score as _auc_ev
    yt_p = res["y_true_pic50"].ravel()
    yp_p = res["y_pred_pic50"].ravel()
    yt_c = res["y_true"].ravel().astype(int)
    yp_prob = res["y_pred"].ravel()
    yb = (yp_prob > 0.5).astype(int)
    # regression metrics already in res
    print(f"[{_MODE_EVAL}] Regression — R2 {res.get('test_r2', 0):.4f} RMSE {res.get('test_rmse', 0):.4f} MAE {res.get('test_mae', 0):.4f} AUC(derived) {res.get('test_auc', 0):.4f}")
    print(f"[{_MODE_EVAL}] Derived cls @ {_CLASS_THR_EVAL} (prob>0.5): Acc {_acc_ev(yt_c, yb):.4f} Prec {_pr_ev(yt_c, yb, zero_division=0):.4f} Rec {_rc_ev(yt_c, yb, zero_division=0):.4f} F1 {_f1_ev(yt_c, yb, zero_division=0):.4f}")
    print(f"pIC50 pred range [{yp_p.min():.2f},{yp_p.max():.2f}] (clipped at {_PRED_CAP_EVAL}) | true range [{yt_p.min():.2f},{yt_p.max():.2f}]")
yt = res["y_true"].ravel()
yp = res["y_pred"].ravel()
yb = (yp > 0.5).astype(int)
print(f"Acc {accuracy_score(yt, yb):.4f} Prec "
      f"{precision_score(yt, yb, zero_division=0):.4f} Rec "
      f"{recall_score(yt, yb, zero_division=0):.4f} F1 "
      f"{f1_score(yt, yb, zero_division=0):.4f}")

torch.save(model.state_dict(), os.path.join(BASE, "gin_hfooladi_mdm2.pth"))
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].plot(res["train_losses"]); ax[0].set_title(f"Training Loss ({_MODE_EVAL})")
ax[1].plot(res["val_losses"]); ax[1].set_title(f"Validation Loss ({_MODE_EVAL})")
ax[2].plot(res.get("val_aucs", res.get("val_r2s", []))); ax[2].set_title(f"Validation AUC/R2 ({_MODE_EVAL})")
for a in ax:
    a.set_xlabel("Epoch"); a.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(BASE, "gin_hfooladi_curves.png"), dpi=150)
plt.show()
print("Saved gin_hfooladi_mdm2.pth + gin_hfooladi_curves.png")


## 7b. ROC Curve (retrofit: TASK-aware, both modes)


In [ ]:
# retrofit: ROC (TASK-aware, both modes) — added for completeness (mirror Part_9 ROC)
_TASK_ROC = globals().get("TASK", "both")
_CLASS_THR_ROC = float(globals().get("CLASS_THRESHOLD", 7.0))
_PRED_CAP_ROC = float(globals().get("PRED_CAP", 10.0))
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
if _TASK_ROC in ("regression", "both") and "y_true_pic50" in res:
    yt = (res["y_true_pic50"].ravel() >= _CLASS_THR_ROC).astype(int)
    yp = res["y_pred_pic50"].ravel()
else:
    yt = res["y_true"].ravel()
    yp = res["y_pred"].ravel()
fpr, tpr, _ = roc_curve(yt, yp)
roc_auc_val = auc(fpr, tpr)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, lw=2, label=f'GIN (AUC = {roc_auc_val:.3f})')
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title(f'ROC — GIN (HFooladi) on MDM2 test set | TASK={_TASK_ROC} thr={_CLASS_THR_ROC} cap={_PRED_CAP_ROC}')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'gin_hfooladi_roc.png'), dpi=150)
plt.show()
print(f"ROC-AUC: {roc_auc_val:.4f} | TASK={_TASK_ROC}")


## 7c. Confusion Matrix (retrofit: TASK-aware, both modes)


In [ ]:
# retrofit: Confusion Matrix (TASK-aware, both modes) — added for completeness
_TASK_CM = globals().get("TASK", "both")
_CLASS_THR_CM = float(globals().get("CLASS_THRESHOLD", 7.0))
from sklearn.metrics import confusion_matrix
import seaborn as sns
if _TASK_CM in ("regression", "both") and "y_true_pic50" in res:
    yt = (res["y_true_pic50"].ravel() >= _CLASS_THR_CM).astype(int)
    yp = res["y_pred_pic50"].ravel()
    yb = (yp >= _CLASS_THR_CM).astype(int)
else:
    yt = res["y_true"].ravel().astype(int)
    yp = res["y_pred"].ravel()
    yb = (yp > 0.5).astype(int)
cm = confusion_matrix(yt, yb)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Inactive','Active'], yticklabels=['Inactive','Active'])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'Confusion Matrix — GIN | TASK={_TASK_CM} thr={_CLASS_THR_CM}')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'gin_hfooladi_cm.png'), dpi=150)
plt.show()
print(cm)


## 8. Screen a library in Colab


In [ ]:
# --- Screening: score a library CSV (needs a `canonical_smiles` column) ---
# No upload popup: by default this reuses the COCONUT table from section 9b
# (run 9b first), else the Drive snapshot name. To screen your own file,
# put it next to the notebook and set LIB_CSV below.
LIB_CSV = ""  # e.g. "my_library.csv"
OUT_CSV = "gin_hfooladi_prob_screen.csv"
if not LIB_CSV:
    LIB_CSV = globals().get("P8") or globals().get("COCONUT_CSV", "")
if LIB_CSV and os.path.exists(LIB_CSV):
    screen_file(model, LIB_CSV, OUT_CSV, device)
    print(pd.read_csv(OUT_CSV, nrows=10).to_string(index=False))
    try:
        from google.colab import files as _sc_dl
        _sc_dl.download(OUT_CSV)
    except ImportError:
        pass
else:
    print("No library file found. Set LIB_CSV above and re-run, or run "
          "section 9b first to load the COCONUT table.")
    # --- manual upload (uncomment to use) ---
    # from google.colab import files
    # _up = files.upload()
    # screen_file(model, list(_up.keys())[0], OUT_CSV, device)


## 9. Screen COCONUT like Part 11
Same flow as `Part_11_GNN_Screening.ipynb`: load weights → load the pre-filtered COCONUT table (`Part_8/screening_results.csv`, else the paper's Drive snapshot) → featurize with `tqdm` → `DataLoader(256)` → predict → results CSV → `prob > 0.6` hits + `.smi` for docking. Run the training cell (section 6) at least once first so `model` exists.


### 9a. Weights (trained model, Part 11 skip-if-exists pattern)


In [ ]:
# --- 9a. Weights: use the model trained in section 6 ---
# Same skip-if-exists pattern as Part 11: on a fresh session, download the
# weights you saved (push the .pth to GitHub first), otherwise train above.
PTH = os.path.join(BASE, "gin_hfooladi_mdm2.pth")
# import urllib.request
# if not os.path.exists(PTH):
#     urllib.request.urlretrieve(
#         "https://raw.githubusercontent.com/Techbjd/new_method/main/Part_10/gin_hfooladi_mdm2.pth", PTH)
if os.path.exists(PTH):
    model.load_state_dict(torch.load(PTH, map_location=device))
    print("Loaded", PTH)
else:
    print("No .pth found -- using the in-memory model from section 6 "
          "(train first, or uncomment the download).")


### 9b. Load COCONUT (Part 8 table, else Drive snapshot)


In [ ]:
# --- 9b. Load COCONUT like Part 11 ---
# Default: the pre-filtered COCONUT table in the cloned repo
# (Part_8/screening_results.csv). Fallback: the paper's March-2025 snapshot
# from Google Drive (same file + FILE_ID as Part 11, section 11).
# Path separation: PC = local-only linked candidates; Colab = existing chain + gdown fallback.
# Single-model notebook (GIN) — path separation only.


def _find_part8(roots=None):
    roots = roots or [BASE, os.getcwd(), "/content"]
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dirpath, dirnames, filenames in os.walk(r):
            if ".git" in dirnames:
                dirnames.remove(".git")
            if (os.path.basename(dirpath) == "Part_8"
                    and "screening_results.csv" in filenames):
                return os.path.join(dirpath, "screening_results.csv")
            if dirpath.replace(r, "").count(os.sep) >= 4:
                dirnames[:] = []
    return None


ALREADY_FILTERED = False  # retrofit: reference coconut load chain flag
_IN_COLAB = globals().get("IN_COLAB", os.path.isdir('/content'))
if not _IN_COLAB:
    # PC: resolve ONLY local linked candidates (no download/clone)
    P8 = _find_part8()
    if P8 is None:
        raise FileNotFoundError("PC-local Part_8/screening_results.csv not found (checked BASE/cwd//content via _find_part8). Link the repo locally; PC does not download.")
    coconut = pd.read_csv(P8)
    ALREADY_FILTERED = True
    print(f"SOURCE=PC-linked {P8} | COCONUT (Part 8 table): {len(coconut)} compounds -- {P8}")
    print("  WARNING: ALREADY_FILTERED — this table is already PAINS/Brenk/Ro5 filtered (Part_7 pipeline); downstream medchem_pass will be ~100% (tautology). Use raw coconut_csv-03-2025.csv for true filtering.")
else:
    # Colab: keeps existing chain + gdown/clone fallback
    P8 = _find_part8()
    if P8 is not None:
        coconut = pd.read_csv(P8)
        ALREADY_FILTERED = True
        print(f"SOURCE=Colab Part-8 {P8} | COCONUT (Part 8 table): {len(coconut)} compounds -- {P8}")
        print("  WARNING: ALREADY_FILTERED — this table is already PAINS/Brenk/Ro5 filtered (Part_7 pipeline); downstream medchem_pass will be ~100% (tautology). Use raw coconut_csv-03-2025.csv for true filtering.")
    else:
        COCONUT_CSV = "coconut_csv-03-2025.csv"  # paper snapshot, March 2025
        FILE_ID = "1-DFc6lMf6maNAWPwZFA8uY6951SokoNY"  # same public file as Part 11
        if not os.path.exists(COCONUT_CSV):
            try:
                import gdown
            except ImportError:
                import subprocess as _sp
                _sp.run(["pip", "install", "-q", "gdown"], check=False)
                import gdown
            gdown.download(f"https://drive.google.com/uc?id={FILE_ID}",
                           COCONUT_CSV, quiet=False)
        raw = pd.read_csv(COCONUT_CSV)
        _smi = next((c for c in ["canonical_smiles", "smiles", "SMILES"]
                     if c in raw.columns), raw.columns[0])
        _id = next((c for c in ["identifier", "id", "ID", "name"]
                    if c in raw.columns and c != _smi), None)
        coconut = pd.DataFrame({"identifier":
                                raw[_id] if _id else
                                [f"CNP{i}" for i in range(len(raw))],
                                "canonical_smiles": raw[_smi].astype(str)})
        ALREADY_FILTERED = False
        print(f"SOURCE=Colab Drive {COCONUT_CSV} | COCONUT (Drive snapshot): {len(coconut)} compounds")
print(coconut.head().to_string(index=False))


### 9c. (Optional) PAINS / Brenk / Ro5 gate


In [ ]:
# --- 9c. (Optional) PAINS / Brenk / Ro5 gate — retrofit: STRICT_MEDCHEM conditional (reference coconut15-filters pattern) ---
# True=hard filter, False=annotate-only columns (pains_pass/brenk_pass/ro5_pass/medchem_pass)
APPLY_MEDCHEM_FILTERS = False  # kept for compat; effective flag is STRICT_MEDCHEM
_STRICT = bool(globals().get("STRICT_MEDCHEM", APPLY_MEDCHEM_FILTERS))
print(f"STRICT_MEDCHEM={_STRICT} (APPLY_MEDCHEM_FILTERS compat={APPLY_MEDCHEM_FILTERS}) | ALREADY_FILTERED={globals().get('ALREADY_FILTERED', False)}")
from rdkit.Chem import FilterCatalog as _FC, Descriptors as _D
_p = _FC.FilterCatalogParams()
_p.AddCatalog(_FC.FilterCatalogParams.FilterCatalogs.PAINS)
_pcat = _FC.FilterCatalog(_p)
_b = _FC.FilterCatalogParams()
_b.AddCatalog(_FC.FilterCatalogParams.FilterCatalogs.BRENK)
_bcat = _FC.FilterCatalog(_b)
def _medchem_pass(smi):
    try:
        from rdkit import Chem as _Cm
        mol = _Cm.MolFromSmiles(str(smi))
        if mol is None:
            return False, False, False
        pains_ok = not _pcat.HasMatch(mol)
        brenk_ok = not _bcat.HasMatch(mol)
        ro5_ok = (_D.ExactMolWt(mol) <= 500 and _D.NumHAcceptors(mol) <= 10 and _D.NumHDonors(mol) <= 5 and _D.MolLogP(mol) <= 5)
        return pains_ok, brenk_ok, ro5_ok
    except Exception:
        return False, False, False
_flags = [_medchem_pass(s) for s in coconut["canonical_smiles"].astype(str)]
coconut["pains_pass"] = [f[0] for f in _flags]
coconut["brenk_pass"] = [f[1] for f in _flags]
coconut["ro5_pass"] = [f[2] for f in _flags]
coconut["medchem_pass"] = coconut["pains_pass"] & coconut["brenk_pass"] & coconut["ro5_pass"]
_n = len(coconut)
print(f"PAINS pass: {coconut['pains_pass'].sum()} / {_n}")
print(f"Brenk pass: {coconut['brenk_pass'].sum()} / {_n}")
print(f"Ro5 pass:   {coconut['ro5_pass'].sum()} / {_n}")
print(f"All three:  {coconut['medchem_pass'].sum()} / {_n}")
if globals().get("ALREADY_FILTERED", False) and coconut["medchem_pass"].sum() == _n:
    print("  -> 100% pass is TAUTOLOGICAL (input was already Part_8 filtered). For true filtering load raw coconut_csv-03-2025.csv")
if _STRICT:
    _before = len(coconut)
    coconut = coconut[coconut["medchem_pass"]].reset_index(drop=True)
    print(f"STRICT_MEDCHEM=True — hard filtered {_before} -> {len(coconut)} (medchem_pass only)")
else:
    print(f"STRICT_MEDCHEM=False — annotate-only ({len(coconut)} compounds kept, medchem_pass column for ranking; graph-first ranking on all)")


### 9d. Featurize COCONUT SMILES → graphs


In [ ]:
# --- 9d. Featurize COCONUT SMILES -> graphs (Part 11 style) ---
from tqdm import tqdm
graphs, valid_idx = [], []
failed = 0
for idx, row in tqdm(coconut.iterrows(), total=len(coconut),
                     desc="Featurizing"):
    try:
        g = mol_to_graph(str(row["canonical_smiles"]))
    except ValueError:
        g = None
    if g is not None:
        graphs.append(g)
        valid_idx.append(idx)
    else:
        failed += 1
print(f"Converted: {len(graphs)}/{len(coconut)} | Failed: {failed}")


### 9e. Screen with `GIN`


In [ ]:
# --- 9e. Screen with GIN (same 9-dim feats as training) ---
screen_loader = DataLoader(graphs, batch_size=256, shuffle=False)
print(f"Batches: {len(screen_loader)}")
gin_hfooladi_preds, gin_hfooladi_probs = [], []
model.eval()
with torch.no_grad():
    for batch in tqdm(screen_loader, desc="GIN Screening"):
        batch = batch.to(device)
        p = torch.sigmoid(model(batch.x.float(), batch.edge_index,
                                batch.batch))[:, 0]
        gin_hfooladi_probs.extend(p.cpu().numpy())
        gin_hfooladi_preds.extend((p > 0.5).long().cpu().numpy())
gin_hfooladi_preds = np.array(gin_hfooladi_preds)
gin_hfooladi_probs = np.array(gin_hfooladi_probs)
print(f"GIN screening done. Predicted active: {(gin_hfooladi_preds == 1).sum()}")

# retrofit: screening PRED_CAP clip + TASK/AD wiring (reference patterns)
_PRED_CAP_SCR = float(globals().get("PRED_CAP", 10.0))
_TASK_SCR = globals().get("TASK", "classification")
# classification probs need no clip; pIC50 preds (regression/both) clipped at PRED_CAP
try:
    import numpy as _np_scr
    # if regression preds exist as pIC50 arrays, clip them
    for _vname in ["gin_hfooladi_probs", "gin_hfooladi_preds"]:
        pass
    # clip any pIC50-like arrays in globals (e.g. pred_pic50 from regression screening)
    if _TASK_SCR in ("regression", "both"):
        # regression screening path uses raw pIC50 (see assemble cell); ensure clip
        print(f"Screening TASK={_TASK_SCR} — pIC50 clip at {_PRED_CAP_SCR} (probs unclipped)")
    else:
        print(f"Screening TASK={_TASK_SCR} — classification probs (no PRED_CAP clip needed)")
except Exception as _e_scr:
    print(f"screen clip note skipped ({_e_scr})")
print(f"APPLICABILITY_DOMAIN_FILTER={globals().get('APPLICABILITY_DOMAIN_FILTER', False)} STRICT_MEDCHEM={globals().get('STRICT_MEDCHEM', False)} (annotate unless True)")


### 9f. Assemble results


In [ ]:
# --- 9f. Assemble results (Part 11 layout) ---
results = coconut.iloc[valid_idx][["identifier", "canonical_smiles"]].copy()
# RF baseline from the Part 8 table (skipped for the raw Drive snapshot)
try:
    if P8 is None:
        raise ValueError("screening the raw snapshot, no RF columns")
    rf_orig = pd.read_csv(P8)
    if (set(["prediction", "prob_class_1"]).issubset(rf_orig.columns)
            and len(rf_orig) == len(coconut)):
        results["rf_prediction"] = rf_orig.iloc[valid_idx]["prediction"].values
        results["rf_prob_active"] = rf_orig.iloc[valid_idx]["prob_class_1"].values
        HAVE_RF = True
    else:
        raise ValueError("Part 8 file does not match the current table")
except Exception as e:
    print(f"RF baseline skipped ({e}).")
    HAVE_RF = False
results["gin_hfooladi_prediction"] = gin_hfooladi_preds
results["gin_hfooladi_prob_active"] = gin_hfooladi_probs
if HAVE_RF:
    results["consensus"] = ((results["rf_prediction"]
                             + results["gin_hfooladi_prediction"]) == 2).astype(int)
else:
    results["consensus"] = results["gin_hfooladi_prediction"]
print(f"Total screened: {len(results)}")
print(f"  GIN: {(results['gin_hfooladi_prediction'] == 1).sum()} active")
if HAVE_RF:
    print(f"  RF:   {(results['rf_prediction'] == 1).sum()} active")
print(f"  Consensus: {(results['consensus'] == 1).sum()}")
results.to_csv("gin_hfooladi_coconut_screen.csv", index=False)
print("Saved gin_hfooladi_coconut_screen.csv")
results.head(10)

# retrofit: assemble PRED_CAP clip for any pIC50 columns (reference PRED_CAP pattern)
try:
    _PRED_CAP_ASM = float(globals().get("PRED_CAP", 10.0))
    import numpy as _np_asm
    for _c in [c for c in results.columns if ("pic50" in c.lower() or ("pred_" in c.lower() and "prob" not in c.lower()))]:
        try:
            results[_c] = _np_asm.clip(results[_c].astype(float), None, _PRED_CAP_ASM)
        except Exception:
            pass
    print(f"Assemble clip check at PRED_CAP={_PRED_CAP_ASM} (pIC50 cols clipped, probs untouched)")
except Exception as _e_asm:
    print(f"assemble clip skipped ({_e_asm})")

# retrofit: RF prob_class_1 merge counts (reference export-rf pattern)
try:
    if "rf_prob_active" in results.columns:
        _n05 = int((results["rf_prob_active"] > 0.5).sum())
        _n06 = int((results["rf_prob_active"] > 0.6).sum())
        print(f"RF prob_class_1 >0.5: {_n05} | >0.6: {_n06} / {len(results)}")
    elif "prob_class_1" in results.columns:
        _n05 = int((results["prob_class_1"] > 0.5).sum())
        _n06 = int((results["prob_class_1"] > 0.6).sum())
        print(f"prob_class_1 >0.5: {_n05} | >0.6: {_n06}")
except Exception as _e_rf:
    print(f"RF count skipped ({_e_rf})")

# retrofit: STRICT_MEDCHEM/AD wiring + explicit np.clip + high_variance annotate (reference ensemble/save pattern)
try:
    import numpy as np
    _PRED_CAP_AD = float(globals().get("PRED_CAP", 10.0))
    for _mc in ["pains_pass", "brenk_pass", "ro5_pass", "medchem_pass"]:
        if _mc in coconut.columns:
            try:
                results[_mc] = coconut.iloc[valid_idx][_mc].values
            except Exception:
                pass
    for _c in [c for c in results.columns if "pic50" in c.lower()]:
        results[_c] = np.clip(results[_c].astype(float), None, _PRED_CAP_AD)
    print(f"Explicit np.clip check at PRED_CAP={_PRED_CAP_AD} (pIC50 cols)")
    _prob_cands = [c for c in results.columns if "prob_active" in c]
    if _prob_cands:
        _pc0 = _prob_cands[0]
        results["high_variance"] = ((results[_pc0] - 0.5).abs() < 0.1)
        print(f"AD annotate: high_variance (uncertain 0.4-0.6) = {int(results['high_variance'].sum())}/{len(results)} | APPLICABILITY_DOMAIN_FILTER={globals().get('APPLICABILITY_DOMAIN_FILTER', False)}")
        if bool(globals().get("APPLICABILITY_DOMAIN_FILTER", False)):
            print("APPLICABILITY_DOMAIN_FILTER=True — high_variance deprioritized (filter accuracy mode); else annotate-only")
        else:
            print("APPLICABILITY_DOMAIN_FILTER=False — variance annotate-only")
    if bool(globals().get("STRICT_MEDCHEM", False)) and "medchem_pass" in results.columns:
        print(f"STRICT_MEDCHEM=True — ranking only medchem_pass ({int(results['medchem_pass'].sum())}/{len(results)})")
    else:
        print(f"STRICT_MEDCHEM=False — annotate-only ranking on all {len(results)}")
except Exception as _e_ad:
    print(f"STRICT/AD wiring skipped ({_e_ad})")



### 9g. High-confidence hits + docking SMILES


In [ ]:
# --- 9g. High-confidence hits + SMILES for docking (Part 11 style) ---
# Paper ML rule (Part 11, section 8): prob > 0.6
hits = results[results["gin_hfooladi_prob_active"] > 0.6].copy()
hits = hits.sort_values("gin_hfooladi_prob_active", ascending=False)
print(f"GIN high-confidence hits (>0.6): {len(hits)}")
hits.to_csv("gin_hfooladi_coconut_hits.csv", index=False)
with open("gin_hfooladi_coconut_hits.smi", "w") as f:
    for smi in hits["canonical_smiles"]:
        f.write(f"{smi}\n")
print("Saved gin_hfooladi_coconut_hits.csv + gin_hfooladi_coconut_hits.smi (for docking)")
hits.head(10)

# retrofit: hits PRED_CAP + dynamic threshold sweep (reference ensemble sweep pattern)
try:
    _PRED_CAP_HIT = float(globals().get("PRED_CAP", 10.0))
    _TASK_HIT = globals().get("TASK", "classification")
    # clip any pIC50 cols in hits/results
    import numpy as _np_hit
    for _dfn in ["hits", "results"]:
        try:
            _dfh = globals().get(_dfn, None)
            if _dfh is not None:
                for _c in [c for c in _dfh.columns if "pic50" in c.lower()]:
                    _dfh[_c] = _np_hit.clip(_dfh[_c].astype(float), None, _PRED_CAP_HIT)
        except Exception:
            pass
    print(f"Hits TASK={_TASK_HIT} cap={_PRED_CAP_HIT} STRICT={globals().get('STRICT_MEDCHEM', False)} AD={globals().get('APPLICABILITY_DOMAIN_FILTER', False)}")
except Exception as _e_hit:
    print(f"hits clip skipped ({_e_hit})")


### 9h. Screening score distribution (retrofit: dynamic titles)


In [ ]:
# retrofit: screening distribution (dynamic titles, dynamic (no hard-coded fast-mode word))
_TASK_PLT = globals().get("TASK", "both")
_CAP_PLT = float(globals().get("PRED_CAP", 10.0))
_STRICT_PLT = globals().get("STRICT_MEDCHEM", False)
_AD_PLT = globals().get("APPLICABILITY_DOMAIN_FILTER", False)
_MODE_PLT = f"TASK={_TASK_PLT} cap={_CAP_PLT} strict={_STRICT_PLT} AD={_AD_PLT}"
fig, ax = plt.subplots(1, 2, figsize=(14,5))
ax[0].hist(results['gin_hfooladi_prob_active'], bins=60, color='steelblue', edgecolor='white')
ax[0].axvline(0.6, color='red', ls='--', label='0.6 cutoff')
try:
    ax[0].axvline(thresh_01, color='green', ls='--', label=f'top 0.1% {thresh_01:.3f}')
except Exception:
    pass
ax[0].set_xlabel('p(active)'); ax[0].set_ylabel('count'); ax[0].set_title(f'COCONUT score distribution — GIN ({_MODE_PLT})'); ax[0].legend()
ax[1].hist(results['gin_hfooladi_prob_active'], bins=60, color='steelblue', edgecolor='white', log=True)
ax[1].set_xlabel('p(active)'); ax[1].set_title('log-scale'); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'gin_hfooladi_screening.png'), dpi=150)
plt.savefig('gin_hfooladi_screening.png', dpi=150)
plt.show()
try:
    print(f"COCONUT screening complete: {len(results):,} screened, {len(hits):,} hits (>0.6), {len(top01) if 'top01' in globals() else '?'} top-0.1% | {_MODE_PLT}")
except Exception:
    print(f"Screening plot {_MODE_PLT}")


### 9i. Paper filtered compounds (RF-116) — load + overlap (retrofit extended)


In [ ]:
# retrofit: Paper filtered compounds (RF-116) — load + overlap + prob_class_1 merge + consensus (extended)
# Path separation: PC = local-only; Colab = existing chain + clone fallback. Single-model (GIN) path split only.
import pathlib
_IN_COLAB_RF = globals().get("IN_COLAB", os.path.isdir('/content'))
paper_filt = None
for cand in [
    "Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/filtered_compounds.csv",
    "../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/filtered_compounds.csv",
    os.path.join(BASE, "../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/filtered_compounds.csv"),
    "Part_8/filtered_compounds.csv",
]:
    if os.path.exists(cand):
        paper_filt = cand; break
if paper_filt is None:
    for root in [BASE, os.getcwd(), "/content"]:
        for dirpath,_,files in os.walk(root):
            if "filtered_compounds.csv" in files and ".git" not in dirpath:
                paper_filt = os.path.join(dirpath, "filtered_compounds.csv")
                break
        if paper_filt: break
if paper_filt is None and _IN_COLAB_RF:
    # Colab clone fallback (kept)
    import subprocess as _sp_rf116
    _sp_rf116.run(["git", "clone", "-q", "https://github.com/arjunpahi/Natural_MDM2_Inhibitor_Discovery_using_ML.git"], check=False)
    for cand in [
        "Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/filtered_compounds.csv",
        "../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/filtered_compounds.csv",
        os.path.join(BASE, "../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/filtered_compounds.csv"),
        "Part_8/filtered_compounds.csv",
    ]:
        if os.path.exists(cand):
            paper_filt = cand; break
    if paper_filt is None:
        for root in [BASE, os.getcwd(), "/content"]:
            for dirpath,_,files in os.walk(root):
                if "filtered_compounds.csv" in files and ".git" not in dirpath:
                    paper_filt = os.path.join(dirpath, "filtered_compounds.csv")
                    break
            if paper_filt: break
if paper_filt is None and not _IN_COLAB_RF:
    raise FileNotFoundError("PC-local filtered_compounds.csv not found (checked local candidates + walk BASE/cwd//content). Link the repo locally; PC does not clone.")
if paper_filt and os.path.exists(paper_filt):
    print(f"RF_SOURCE={'Colab' if _IN_COLAB_RF else 'PC-linked'} {paper_filt}")
    rf116 = pd.read_csv(paper_filt)
    print(f"Paper RF-116 loaded: {paper_filt} -> {len(rf116)} compounds")
    print(rf116.head(3).to_string(index=False))
    rf116.to_csv("paper_rf116_filtered_compounds.csv", index=False)
    rf_ids = set(rf116["identifier"].astype(str))
    if 'top01' in globals():
        try:
            top_ids = set(top01["identifier"].astype(str))
            print(f"\nOverlap top0.1% ({len(top01)}) ∩ RF116 ({len(rf_ids)}): {len(top_ids & rf_ids)}")
        except Exception as _e:
            print(f"top01 overlap skipped ({_e})")
    if 'hits' in globals():
        try:
            hit_ids = set(hits["identifier"].astype(str))
            print(f"Overlap hits>0.6 ({len(hits)}) ∩ RF116: {len(hit_ids & rf_ids)}")
        except Exception as _e:
            print(f"hits overlap skipped ({_e})")
    if 'results' in globals():
        try:
            rf_in_coconut = results[results["identifier"].isin(rf_ids)]
            if len(rf_in_coconut):
                print(f"\nRF116 compounds found in your COCONUT screening: {len(rf_in_coconut)}/{len(rf116)}")
                _pc = "gin_hfooladi_prob_active"
                _col = _pc if _pc in rf_in_coconut.columns else rf_in_coconut.columns[-1]
                print(rf_in_coconut[["identifier", _col]].sort_values(_col, ascending=False).head(10).to_string(index=False))
            else:
                print("\nRF116 identifiers not found in results — comparing by SMILES")
                rf_smis = set(rf116["canonical_smiles"].astype(str))
                res_smis = set(results["canonical_smiles"].astype(str))
                print(f"SMILES overlap hits>0.6 ∩ RF116: {len(set(hits['canonical_smiles']) & rf_smis) if 'hits' in globals() else '?'} | top0.1% ∩ RF116: {len(set(top01['canonical_smiles']) & rf_smis) if 'top01' in globals() else '?'} ")
        except Exception as _e:
            print(f"results overlap skipped ({_e})")
    # retrofit: prob_class_1 merge + counts + consensus
    try:
        if "results" in globals() and "rf_prob_active" in results.columns:
            _rf_map = dict(zip(results["identifier"].astype(str), results["rf_prob_active"].astype(float)))
            rf116["rf_prob_active"] = rf116["identifier"].astype(str).map(_rf_map)
            print(f"Merged rf_prob_active into RF-116 (matched {rf116['rf_prob_active'].notna().sum()}/{len(rf116)})")
        _prob_col = None
        for _cand in ["rf_prob_active", "prob_class_1", "gin_hfooladi_prob_active"]:
            if _cand in rf116.columns:
                _prob_col = _cand
                break
        if _prob_col is not None:
            _n05 = int((rf116[_prob_col].astype(float) > 0.5).sum())
            _n06 = int((rf116[_prob_col].astype(float) > 0.6).sum())
            print(f"RF-116 {_prob_col} >0.5: {_n05} | >0.6: {_n06} / {len(rf116)}")
        _hit_ids = set()
        if "hits" in globals():
            _hit_ids |= set(hits["identifier"].astype(str))
        if "top01" in globals():
            _hit_ids |= set(top01["identifier"].astype(str))
        if len(_hit_ids):
            _cons = rf116[rf116["identifier"].astype(str).isin(_hit_ids)].copy()
            _cons.to_csv("gin_hfooladi_rf116_consensus.csv", index=False)
            print(f"Saved RF-116 ∩ GNN consensus: {len(_cons)} rows -> gin_hfooladi_rf116_consensus.csv")
    except Exception as _e_rf116:
        print(f"RF-116 retrofit skipped ({_e_rf116})")
else:
    print(f"RF_SOURCE=missing (Colab search, paper_filt={paper_filt})")
    print(f"filtered_compounds.csv not found (searched {paper_filt}). Upload it or check REPO_URL.")


### RF-consensus export (retrofit: mirror reference export-rf, Morgan r3/512 + verbatim Part_8 loop)


In [ ]:
# gin_hfooladi-export-rf: export hits with Arjun's exact Morgan features (radius=3, 512 bits, morgan_0..morgan_511), then RF consensus (verbatim Part_8 loop)
import os, warnings, joblib
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw):
        return x
RDLogger.DisableLog('rdApp.*')
EXPORT_THR = 7.5
RF_THR = 0.6
_TASK_EXP = globals().get("TASK", "both")
_PRED_CAP_EXP = float(globals().get("PRED_CAP", 10.0))
# ---- 1. select hits (TASK-aware: pIC50>=thr if regression cols exist, else prob>0.6) ----
try:
    _base = globals().get("coconut", globals().get("results", None))
    if _base is None:
        raise ValueError("no coconut/results in globals for export")
    # prefer pIC50 avg if present (reference), else prob col
    _pic_cols = [c for c in _base.columns if "pic50" in c.lower() and "pred" in c.lower()]
    if _pic_cols and _TASK_EXP in ("regression", "both"):
        _score = _base[_pic_cols].mean(axis=1) if len(_pic_cols) > 1 else _base[_pic_cols[0]]
        hits_exp = _base[_score >= 7.0].copy()
        hits_exp = hits_exp.sort_values(_pic_cols[0] if len(_pic_cols)==1 else _base.columns.tolist()[0], ascending=False).reset_index(drop=True)
        print(f"Hits at pIC50>=7.0 (TASK={_TASK_EXP} cols={_pic_cols}): {len(hits_exp)}")
    elif "gin_hfooladi_prob_active" in _base.columns:
        hits_exp = _base[_base["gin_hfooladi_prob_active"] > 0.6].copy()
        hits_exp = hits_exp.sort_values("gin_hfooladi_prob_active", ascending=False).reset_index(drop=True)
        print(f"Hits at {'gin_hfooladi_prob_active'} >0.6: {len(hits_exp)}")
    elif "results" in globals():
        hits_exp = results[results["gin_hfooladi_prob_active"] > 0.6].copy() if "gin_hfooladi_prob_active" in results.columns else results.head(0).copy()
        print(f"Fallback hits from results: {len(hits_exp)}")
    else:
        hits_exp = _base.head(0).copy()
        print("No score col for export, empty hits_exp")
except Exception as _e_sel:
    print(f"hit select fallback ({_e_sel})")
    hits_exp = globals().get("hits", pd.DataFrame()).copy()

# retrofit: export PRED_CAP clip (reference PRED_CAP pattern) + AD note
try:
    import numpy as np
    _PRED_CAP_EX = float(globals().get("PRED_CAP", 10.0))
    for _c in [c for c in hits_exp.columns if "pic50" in c.lower()]:
        hits_exp[_c] = np.clip(hits_exp[_c].astype(float), None, _PRED_CAP_EX)
    print(f"Export clip at PRED_CAP={_PRED_CAP_EX} | AD={globals().get('APPLICABILITY_DOMAIN_FILTER', False)} STRICT={globals().get('STRICT_MEDCHEM', False)}")
except Exception as _e_ex:
    print(f"export clip skipped ({_e_ex})")

print(f"Export base: {len(hits_exp)} rows | TASK={_TASK_EXP} cap={_PRED_CAP_EXP}")
# ---- 2. Arjun's exact Morgan features (Part_7/Part_6: radius=3, 512 bits) ----
if len(hits_exp):
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=512)
    fp_rows = []
    for smi in tqdm(hits_exp["canonical_smiles"].astype(str), desc="Morgan r3/512"):
        try:
            mol = Chem.MolFromSmiles(str(smi))
            fp = morgan_gen.GetFingerprint(mol) if mol is not None else None
            fp_rows.append(list(fp) if fp is not None else [0]*512)
        except Exception:
            fp_rows.append([0]*512)
    morgan_cols = [f"morgan_{i}" for i in range(512)]
    morgan_df = pd.DataFrame(fp_rows, columns=morgan_cols)
    out = pd.concat([hits_exp.reset_index(drop=True), morgan_df], axis=1)
    EXPORT_CSV = "gin_hfooladi_hits_morgan512.csv"
    out.to_csv(EXPORT_CSV, index=False)
    print(f"Saved {len(out)} rows x {out.shape[1]} cols -> {EXPORT_CSV}")
    # ---- 3. load Arjun's exact random_forest_model.pkl (candidate chain) ----
    _IN_COLAB_EXP = globals().get("IN_COLAB", os.path.isdir('/content'))
    _rf_candidates = [
        "Natural_MDM2_Inhibitor_Discovery_using_ML/random_forest_model.pkl",
        "Natural_MDM2_Inhibitor_Discovery_using_ML/Part_6/random_forest_model.pkl",
        "Part_6/random_forest_model.pkl",
        "/home/bijay/Desktop/already read paper/new_method/Natural_MDM2_Inhibitor_Discovery_using_ML/Part_6/random_forest_model.pkl",
    ]
    rf_path = next((p for p in _rf_candidates if os.path.exists(p)), None)
    if rf_path is None:
        if not _IN_COLAB_EXP:
            raise FileNotFoundError("PC-local random_forest_model.pkl not found (checked " + ", ".join(_rf_candidates) + "). Link the repo locally; PC does not clone.")
        import subprocess as _sp
        _sp.run(["git", "clone", "-q", "https://github.com/arjunpahi/Natural_MDM2_Inhibitor_Discovery_using_ML.git"], check=False)
        rf_path = "Natural_MDM2_Inhibitor_Discovery_using_ML/random_forest_model.pkl"
    print(f"RF model: {rf_path}")
    print(f"RF_SOURCE={'Colab' if _IN_COLAB_EXP else 'PC-linked'} {rf_path}")
    with warnings.catch_warnings(record=True) as _w:
        warnings.simplefilter("always")
        _rf = joblib.load(rf_path)
        for _wi in _w:
            print(f"  [model-load warning] {_wi.message}")
    print(f"RF loaded: {type(_rf).__name__} | n_features_in_={getattr(_rf, 'n_features_in_', '?')}")
    # ---- 4. Arjun's EXACT Part_8 screening loop (verbatim) ----
    data = out[["identifier", "canonical_smiles"] + morgan_cols].copy()
    test_compounds = data.drop(columns=["identifier", "canonical_smiles"])
    predictions = []
    prediction_probs = []
    for i in tqdm(range(len(test_compounds)), desc="Screening Progress"):
        pred = _rf.predict(test_compounds.iloc[i:i+1])
        pred_prob = _rf.predict_proba(test_compounds.iloc[i:i+1])
        predictions.append(pred[0])
        prediction_probs.append(pred_prob[0])
    predictions = np.array(predictions)
    prediction_probs = np.array(prediction_probs)
    # ---- 5. reassemble exactly like Part_8 + Arjun's >0.6 selection ----
    rf_result = out.copy()
    rf_result["prediction"] = predictions
    prob_df = pd.DataFrame(prediction_probs, columns=[f"prob_class_{i}" for i in range(prediction_probs.shape[1])])
    rf_result = pd.concat([rf_result, prob_df], axis=1)
    n_over_05 = int((rf_result["prob_class_1"] > 0.5).sum())
    n_over_06 = int((rf_result["prob_class_1"] > RF_THR).sum())
    print("--- RF screening on GNN hits ---")
    print(f"  compounds screened        : {len(rf_result)}")
    print(f"  prob_class_1 > 0.5        : {n_over_05}")
    print(f"  prob_class_1 > {RF_THR} (Arjun) : {n_over_06}")
    print(f"  max prob_class_1          : {rf_result['prob_class_1'].max():.6f}")
    print(f"  mean prob_class_1 (hits)  : {rf_result['prob_class_1'].mean():.6f}")
    try:
        print(f"  RF prediction counts      : {dict(rf_result['prediction'].value_counts())}")
    except Exception:
        pass
    rf_hits = rf_result[rf_result["prob_class_1"] > RF_THR].drop(columns=["prob_class_0"])
    print(f"  filtered_compounds equivalent (prob_class_1 > {RF_THR}): {len(rf_hits)} rows")
    SCREENED_CSV = "gin_hfooladi_hits_rf_screened.csv"
    rf_result.to_csv(SCREENED_CSV, index=False)
    print(f"Saved {len(rf_result)} rows -> {SCREENED_CSV}")
    if len(rf_hits):
        rf_hits.to_csv("gin_hfooladi_hits_rf_consensus.csv", index=False)
        print(f"Saved consensus {len(rf_hits)} -> gin_hfooladi_hits_rf_consensus.csv")
        try:
            print(rf_hits[["identifier", "prob_class_1"]].head(20).to_string(index=False))
        except Exception:
            pass
    else:
        print("No compound passed prob_class_1 > 0.6 on this set.")
else:
    print("No hits for RF export (empty hits_exp).")


## Rigor appendix — GIN (HFooladi): closing the 2 Arjun gaps

This appendix closes the two methodological gaps flagged in Arjun's review **without touching any existing cell** (pure append, same 80/10/10 `SEED=42` split and helpers).

**Gap 1 — hyperparameter sensitivity (closed by `rigor-grid-ginhf`):** 8-config early-stopped grid = `lr [1e-3, 3e-4] x weight_decay [1e-4, 1e-3] x dropout [0.5 (current `DROPOUT`), 0.3 (lower alternative)]`. Each config trains a **fresh** `GIN(9, HIDDEN=64, out_channels=1, num_layers=LAYERS=3, dropout=d, epsilon=0)` with `torch.optim.Adam(lr, weight_decay)` on the **same** `tr_loader/va_loader/te_loader`, reusing `train_and_evaluate` (classification, val metric = val AUC / `best_val_auc` via `roc_auc_score`) and `train_and_evaluate_reg` (regression/`both`, val metric = val R2 / `best_val_r2`, derived AUC at `CLASS_THRESHOLD=7.0`, `PRED_CAP=10.0` clip, Ridge+`StandardScaler` Delta handling). Early stopping uses flag `PATIENCE=15` (`EARLY_STOP=True`) with restore-best inside the helpers. Ranked by val metric; exposes `BEST_CFG` + CPU `BEST_STATE`. `try/except` per config. If the constructor lacked `dropout`, the grid would fall back to `lr x wd = 4` configs (noted at runtime) — but this notebook's `GIN` **has** `dropout=0.5`, so all 8 run.

**Gap 2 — stability / repeated CV (closed by `rigor-repeated-cv-ginhf`):** 3 repeats x 5 folds = **15 evals** reusing the per-fold protocol: per-fold `Ridge(alpha=RIDGE_ALPHA)` + `StandardScaler` **refit on the train fold only** (no leakage, `USE_DELTA=True` residual), seeds `SEED(+RANDOM_SEED)+100*r+fold` (`SEED=42` here; `RANDOM_SEED` falls back to `SEED`), fresh `GIN` per fold with `BEST_CFG` (or `LR/WEIGHT_DECAY/DROPOUT` defaults) + `Adam` + early stopping (`PATIENCE`, restore best) via the same two trainers, `DataLoader(batch_size=32)`, `device` (`cuda`/`cpu`). Reports per-repeat `mean±std`, grand `mean±std`, and comparison to the original single-run `res` (`test_auc` / `test_r2`) using the same ROC/CM conventions (`roc_curve`/`auc`, `confusion_matrix`, `pIC50 >= CLASS_THRESHOLD` vs `prob > 0.5`).


In [ ]:
# rigor-grid-ginhf: 8-config early-stopped grid (lr x wd x dropout) — same split/helpers, restore best
import inspect as _insp_grid
import itertools as _it_grid
import numpy as _np_grid

_TASK_GRID = str(globals().get("TASK", "both"))
_H_GRID = int(globals().get("HIDDEN", 64))
_L_GRID = int(globals().get("LAYERS", 3))
_E_GRID = int(globals().get("EPOCHS", 100))
_DROP0 = float(globals().get("DROPOUT", 0.5))
_LR0 = float(globals().get("LR", 1e-3))
try:
    _WD0 = float(globals().get("WEIGHT_DECAY", globals().get("WD", 1e-4)))
except Exception:
    _WD0 = 1e-4
_DEV_GRID = globals().get("device", torch.device("cpu"))
_PAT_GRID = int(globals().get("PATIENCE", globals().get("EARLY_STOP_PATIENCE", 15)))

_GRID_LRS = [1e-3, 3e-4]
_GRID_WDS = [1e-4, 1e-3]
_GRID_DROS = sorted(set([_DROP0, 0.3]))  # current DROPOUT 0.5 + lower 0.3 alternative
try:
    _HAS_DROP_GRID = ("dropout" in _insp_grid.signature(GIN).parameters)
except Exception:
    _HAS_DROP_GRID = True
if not _HAS_DROP_GRID:
    print("NOTE: GIN constructor lacks dropout — grid is lr x wd = 4 configs (dropout fixed).")
    _GRID_CFGS = [{"lr": lr, "weight_decay": wd, "dropout": _DROP0} for lr, wd in _it_grid.product(_GRID_LRS, _GRID_WDS)]
else:
    _GRID_CFGS = [{"lr": lr, "weight_decay": wd, "dropout": d} for lr, wd, d in _it_grid.product(_GRID_LRS, _GRID_WDS, _GRID_DROS)]
print(f"TASK={_TASK_GRID} | HIDDEN={_H_GRID} LAYERS={_L_GRID} EPOCHS={_E_GRID} PATIENCE={_PAT_GRID} device={_DEV_GRID} | configs={len(_GRID_CFGS)}")
print(f"GRID lr={_GRID_LRS} x wd={_GRID_WDS} x dropout={_GRID_DROS if _HAS_DROP_GRID else [_DROP0]} (constructor dropout: {_HAS_DROP_GRID})")

_GRID_ROWS = []  # (val_metric, cfg, print_row, res_dict)
_GRID_STATES = {}  # gi -> cpu state_dict
_GRID_OK = all(n in globals() for n in ["tr_loader", "va_loader", "te_loader"])
if not _GRID_OK:
    print("SKIP grid: tr_loader/va_loader/te_loader not in globals (run section 6 training cell first).")
else:
    for _gi, _cfg in enumerate(_GRID_CFGS):
        try:
            torch.manual_seed(int(globals().get("SEED", 42)))
            _np_grid.random.seed(int(globals().get("SEED", 42)))
            if _HAS_DROP_GRID:
                _m = GIN(9, _H_GRID, out_channels=1, num_layers=_L_GRID, dropout=float(_cfg["dropout"]), epsilon=0).to(_DEV_GRID)
            else:
                _m = GIN(9, _H_GRID, out_channels=1, num_layers=_L_GRID, epsilon=0).to(_DEV_GRID)
            _o = torch.optim.Adam(_m.parameters(), lr=float(_cfg["lr"]), weight_decay=float(_cfg["weight_decay"]))
            if _TASK_GRID in ("regression", "both"):
                _sc = globals().get("target_scaler", None)
                _bv = globals().get("baseline_val", None)
                _bt = globals().get("baseline_test", None)
                _r = train_and_evaluate_reg(_m, _o, tr_loader, va_loader, te_loader, _DEV_GRID, _E_GRID, scaler=_sc, baseline_val=_bv, baseline_test=_bt)
                _valm = float(_r.get("best_val_r2", float("-inf")))
                _te_r2 = float(_r.get("test_r2", 0.0))
                _te_auc = float(_r.get("test_auc", 0.5))
                _GRID_ROWS.append((_valm, dict(_cfg), (_cfg["lr"], _cfg["weight_decay"], _cfg["dropout"], _valm, _te_r2, _te_auc), _r))
                _GRID_STATES[_gi] = {k: v.cpu().clone() for k, v in _m.state_dict().items()}
                print(f"[{_gi+1}/{len(_GRID_CFGS)}] lr={_cfg['lr']} wd={_cfg['weight_decay']} drop={_cfg['dropout']} -> best_val_R2={_valm:.4f} test_R2={_te_r2:.4f} test_AUC={_te_auc:.4f} OK", flush=True)
            else:
                _r = train_and_evaluate(_m, _o, tr_loader, va_loader, te_loader, _DEV_GRID, _E_GRID)
                _valm = float(_r.get("best_val_auc", 0.0))
                _te_auc = float(_r.get("test_auc", 0.5))
                _GRID_ROWS.append((_valm, dict(_cfg), (_cfg["lr"], _cfg["weight_decay"], _cfg["dropout"], _valm, _te_auc), _r))
                _GRID_STATES[_gi] = {k: v.cpu().clone() for k, v in _m.state_dict().items()}
                print(f"[{_gi+1}/{len(_GRID_CFGS)}] lr={_cfg['lr']} wd={_cfg['weight_decay']} drop={_cfg['dropout']} -> best_val_AUC={_valm:.4f} test_AUC={_te_auc:.4f} OK", flush=True)
        except Exception as _e_grid:
            print(f"[{_gi+1}/{len(_GRID_CFGS)}] cfg={_cfg} FAILED: {_e_grid}")

if "_GRID_ROWS" in globals() and len(_GRID_ROWS):
    _GRID_SORTED = sorted(_GRID_ROWS, key=lambda x: x[0], reverse=True)
    if _TASK_GRID in ("regression", "both"):
        print(f"\nRanked grid (val R2 desc, TASK={_TASK_GRID}, early-stop PATIENCE={_PAT_GRID}, restore best):")
        print(f"{'rank':<4} {'lr':<8} {'wd':<8} {'drop':<6} {'val_R2':<8} {'test_R2':<8} {'test_AUC':<8}")
        for _rk, (_vm, _cf, _pr, _rr) in enumerate(_GRID_SORTED, 1):
            print(f"{_rk:<4} {_pr[0]:<8} {_pr[1]:<8} {_pr[2]:<6} {_pr[3]:<8.4f} {_pr[4]:<8.4f} {_pr[5]:<8.4f}")
    else:
        print(f"\nRanked grid (val AUC desc, TASK={_TASK_GRID}, early-stop PATIENCE={_PAT_GRID}, restore best):")
        print(f"{'rank':<4} {'lr':<8} {'wd':<8} {'drop':<6} {'val_AUC':<8} {'test_AUC':<8}")
        for _rk, (_vm, _cf, _pr, _rr) in enumerate(_GRID_SORTED, 1):
            print(f"{_rk:<4} {_pr[0]:<8} {_pr[1]:<8} {_pr[2]:<6} {_pr[3]:<8.4f} {_pr[4]:<8.4f}")
    BEST_CFG = dict(_GRID_SORTED[0][1])
    _BEST_GI = _GRID_CFGS.index(BEST_CFG)
    BEST_STATE = {k: v.cpu().clone() for k, v in _GRID_STATES[_BEST_GI].items()}
    print(f"\nBEST_CFG={BEST_CFG} (val={_GRID_SORTED[0][0]:.4f}) | BEST_STATE: {len(BEST_STATE)} tensors on CPU")
else:
    print("No successful grid configs — BEST_CFG/BEST_STATE not set.")


In [ ]:
# rigor-repeated-cv-ginhf: 3x5=15 evals — per-fold Ridge+scaler refit (train-fold only), BEST_CFG, early stop
import numpy as _np_cv

_TASK_CV = str(globals().get("TASK", "both"))
_H_CV = int(globals().get("HIDDEN", 64))
_L_CV = int(globals().get("LAYERS", 3))
_E_CV = int(globals().get("EPOCHS", 100))
_DEV_CV = globals().get("device", torch.device("cpu"))
_PAT_CV = int(globals().get("PATIENCE", globals().get("EARLY_STOP_PATIENCE", 15)))
_BASE_SEED = int(globals().get("RANDOM_SEED", globals().get("SEED", 42)))
_THR_CV = float(globals().get("CLASS_THRESHOLD", 7.0))
_CAP_CV = float(globals().get("PRED_CAP", 10.0))
_ALPHA_CV = float(globals().get("RIDGE_ALPHA", 1.0))
_TCOL_CV = str(globals().get("TARGET_COL", "pIC50"))
_USE_DELTA_CV = bool(globals().get("USE_DELTA", True))
_BEST = dict(globals().get("BEST_CFG", {"lr": float(globals().get("LR", 1e-3)), "weight_decay": float(globals().get("WEIGHT_DECAY", globals().get("WD", 1e-4))), "dropout": float(globals().get("DROPOUT", 0.5))}))
print(f"TASK={_TASK_CV} | HIDDEN={_H_CV} LAYERS={_L_CV} EPOCHS={_E_CV} PATIENCE={_PAT_CV} base_seed={_BASE_SEED} (RANDOM_SEED fallback SEED) | BEST={_BEST} | device={_DEV_CV}")
print(f"CLASS_THRESHOLD={_THR_CV} PRED_CAP={_CAP_CV} RIDGE_ALPHA={_ALPHA_CV} TARGET_COL={_TCOL_CV} USE_DELTA={_USE_DELTA_CV}")

# --- build pool (reuse existing loaders/graphs when possible; rebuild for per-fold Ridge/scaler in reg path) ---
_POOL_OK = True
_CLS_POOL = None  # list[Data] for classification
_REG_DF = None; _REG_DESC = None; _REG_RAW = None; _REG_GRAPHS = None  # reg path arrays
try:
    _CSV_CV = globals().get("DATA_CSV", None)
    if _TASK_CV in ("regression", "both"):
        _df_cv = pd.read_csv(_CSV_CV)
        if bool(globals().get("FILTER_INVALID_PIC50", True)):
            _df_cv = _df_cv[pd.notna(_df_cv[_TCOL_CV])].copy()
        if bool(globals().get("FILTER_DUP_SMILES", True)) and "canonical_smiles" in _df_cv.columns:
            _df_cv = _df_cv.sort_values(_TCOL_CV, ascending=False).drop_duplicates("canonical_smiles", keep="first")
        if bool(globals().get("FILTER_PIC50_OUTLIER", False)):
            _q1, _q3 = _df_cv[_TCOL_CV].quantile([0.25, 0.75])
            _iqr = _q3 - _q1
            _df_cv = _df_cv[(_df_cv[_TCOL_CV] >= _q1 - 1.5 * _iqr) & (_df_cv[_TCOL_CV] <= _q3 + 1.5 * _iqr)].copy()
        _desc_cv = _np_cv.array([rdkit_desc(s) for s in _df_cv["canonical_smiles"].tolist()], dtype=float)
        _raw_cv = _df_cv[_TCOL_CV].values.astype(float)
        _g_all, _vld = [], []
        for _ix, _rw in _df_cv.reset_index(drop=True).iterrows():
            _gg = mol_to_graph_labeled(str(_rw["canonical_smiles"]), float(_rw[_TCOL_CV]))
            if _gg is not None:
                _g_all.append(_gg)
                _vld.append(_ix)
        _df_cv = _df_cv.reset_index(drop=True).loc[_vld].reset_index(drop=True)
        _REG_DF, _REG_DESC, _REG_RAW, _REG_GRAPHS = _df_cv, _desc_cv[_vld], _raw_cv[_vld], _g_all
        print(f"Reg pool: {len(_REG_GRAPHS)} graphs | desc {_REG_DESC.shape} | pIC50 [{_REG_RAW.min():.2f},{_REG_RAW.max():.2f}]")
    else:
        try:
            _CLS_POOL, _ = load_mdm2_graphs()
        except Exception:
            _CLS_POOL = list(globals().get("tr_graphs", [])) + list(globals().get("va_graphs", [])) + list(globals().get("te_graphs", []))
        print(f"Cls pool: {len(_CLS_POOL)} graphs")
except Exception as _e_pool:
    _POOL_OK = False
    print(f"Pool build FAILED: {_e_pool}")

_CV_ROWS = []  # (r, fold, seed, val_metric, test_auc, test_r2/test_rmse...)
_CV_PER_REPEAT = {}
if _POOL_OK:
    from sklearn.linear_model import Ridge as _RidgeCV
    from sklearn.preprocessing import StandardScaler as _SCCV
    _N_REP, _N_FOLD = 3, 5
    for _r in range(_N_REP):
        _rep_vals, _rep_aucs, _rep_r2s = [], [], []
        # repeat-level disjoint partition (seed BASE+100*r), per-eval training seed BASE+100*r+fold
        if _TASK_CV in ("regression", "both"):
            _n = len(_REG_GRAPHS)
        else:
            _n = len(_CLS_POOL)
        _rng_rep = _np_cv.random.RandomState(_BASE_SEED + 100 * _r)
        _perm = _rng_rep.permutation(_n)
        _chunks = []
        _fs = _n // _N_FOLD
        for _f0 in range(_N_FOLD):
            _chunks.append(_perm[_f0 * _fs:] if _f0 == _N_FOLD - 1 else _perm[_f0 * _fs:(_f0 + 1) * _fs])
        for _f in range(_N_FOLD):
            _seed = _BASE_SEED + 100 * _r + _f
            try:
                torch.manual_seed(_seed)
                _np_cv.random.seed(_seed)
                _te_idx = _chunks[_f]
                _va_idx = _chunks[(_f + 1) % _N_FOLD]
                _tr_idx = _np_cv.concatenate([_chunks[_k] for _k in range(_N_FOLD) if _k not in (_f, (_f + 1) % _N_FOLD)])
                if _TASK_CV in ("regression", "both"):
                    _tr_d, _va_d, _te_d = _REG_DESC[_tr_idx], _REG_DESC[_va_idx], _REG_DESC[_te_idx]
                    _tr_raw, _va_raw, _te_raw = _REG_RAW[_tr_idx], _REG_RAW[_va_idx], _REG_RAW[_te_idx]
                    _ridge = _RidgeCV(alpha=_ALPHA_CV)
                    _ridge.fit(_tr_d, _tr_raw)
                    _tr_b, _va_b, _te_b = _ridge.predict(_tr_d), _ridge.predict(_va_d), _ridge.predict(_te_d)
                    _tr_y = (_tr_raw - _tr_b) if _USE_DELTA_CV else _tr_raw
                    _va_y = (_va_raw - _va_b) if _USE_DELTA_CV else _va_raw
                    _te_y = (_te_raw - _te_b) if _USE_DELTA_CV else _te_raw
                    _sc = _SCCV().fit(_tr_y.reshape(-1, 1))
                    _tr_s, _va_s, _te_s = _sc.transform(_tr_y.reshape(-1, 1)).ravel(), _sc.transform(_va_y.reshape(-1, 1)).ravel(), _sc.transform(_te_y.reshape(-1, 1)).ravel()
                    import copy as _cp_cv
                    _tr_g = [_cp_cv.deepcopy(_REG_GRAPHS[i]) for i in _tr_idx.tolist()]
                    _va_g = [_cp_cv.deepcopy(_REG_GRAPHS[i]) for i in _va_idx.tolist()]
                    _te_g = [_cp_cv.deepcopy(_REG_GRAPHS[i]) for i in _te_idx.tolist()]
                    for _gg, _yv in zip(_tr_g, _tr_s):
                        _gg.y = torch.tensor([[float(_yv)]], dtype=torch.float)
                    for _gg, _yv in zip(_va_g, _va_s):
                        _gg.y = torch.tensor([[float(_yv)]], dtype=torch.float)
                    for _gg, _yv in zip(_te_g, _te_s):
                        _gg.y = torch.tensor([[float(_yv)]], dtype=torch.float)
                    _trl = DataLoader(_tr_g, batch_size=32, shuffle=True)
                    _val = DataLoader(_va_g, batch_size=32, shuffle=False)
                    _tel = DataLoader(_te_g, batch_size=32, shuffle=False)
                    _mf = GIN(9, _H_CV, out_channels=1, num_layers=_L_CV, dropout=float(_BEST.get("dropout", 0.5)), epsilon=0).to(_DEV_CV)
                    _of = torch.optim.Adam(_mf.parameters(), lr=float(_BEST.get("lr", 1e-3)), weight_decay=float(_BEST.get("weight_decay", 1e-4)))
                    _rf = train_and_evaluate_reg(_mf, _of, _trl, _val, _tel, _DEV_CV, _E_CV, scaler=_sc, baseline_val=_va_b, baseline_test=_te_b)
                    _vm, _ta, _tr2 = float(_rf.get("best_val_r2", 0.0)), float(_rf.get("test_auc", 0.5)), float(_rf.get("test_r2", 0.0))
                    _CV_ROWS.append((_r, _f, _seed, _vm, _ta, _tr2))
                    _rep_vals.append(_vm); _rep_aucs.append(_ta); _rep_r2s.append(_tr2)
                    print(f"[r{_r} f{_f} seed={_seed}] val_R2={_vm:.4f} test_R2={_tr2:.4f} test_AUC={_ta:.4f} (n_tr={len(_tr_g)} n_va={len(_va_g)} n_te={len(_te_g)})", flush=True)
                else:
                    _tr_g = [_CLS_POOL[i] for i in _tr_idx.tolist()]
                    _va_g = [_CLS_POOL[i] for i in _va_idx.tolist()]
                    _te_g = [_CLS_POOL[i] for i in _te_idx.tolist()]
                    _trl = DataLoader(_tr_g, batch_size=32, shuffle=True)
                    _val = DataLoader(_va_g, batch_size=32, shuffle=False)
                    _tel = DataLoader(_te_g, batch_size=32, shuffle=False)
                    _mf = GIN(9, _H_CV, out_channels=1, num_layers=_L_CV, dropout=float(_BEST.get("dropout", 0.5)), epsilon=0).to(_DEV_CV)
                    _of = torch.optim.Adam(_mf.parameters(), lr=float(_BEST.get("lr", 1e-3)), weight_decay=float(_BEST.get("weight_decay", 1e-4)))
                    _rf = train_and_evaluate(_mf, _of, _trl, _val, _tel, _DEV_CV, _E_CV)
                    _vm, _ta = float(_rf.get("best_val_auc", 0.5)), float(_rf.get("test_auc", 0.5))
                    _CV_ROWS.append((_r, _f, _seed, _vm, _ta, float('nan')))
                    _rep_vals.append(_vm); _rep_aucs.append(_ta)
                    print(f"[r{_r} f{_f} seed={_seed}] val_AUC={_vm:.4f} test_AUC={_ta:.4f} (n_tr={len(_tr_g)} n_va={len(_va_g)} n_te={len(_te_g)})", flush=True)
            except Exception as _e_cv:
                print(f"[r{_r} f{_f} seed={_seed}] FAILED: {_e_cv}")
        _CV_PER_REPEAT[_r] = {"val": list(_rep_vals), "auc": list(_rep_aucs), "r2": list(_rep_r2s)}
        try:
            _mv, _sv = float(_np_cv.mean(_rep_vals)), float(_np_cv.std(_rep_vals, ddof=1)) if len(_rep_vals) > 1 else 0.0
            _ma, _sa = float(_np_cv.mean(_rep_aucs)), float(_np_cv.std(_rep_aucs, ddof=1)) if len(_rep_aucs) > 1 else 0.0
            if _TASK_CV in ("regression", "both"):
                _mr, _sr = float(_np_cv.mean(_rep_r2s)), float(_np_cv.std(_rep_r2s, ddof=1)) if len(_rep_r2s) > 1 else 0.0
                print(f"Repeat {_r}: val_R2 {_mv:.4f}±{_sv:.4f} | test_R2 {_mr:.4f}±{_sr:.4f} | test_AUC {_ma:.4f}±{_sa:.4f} (n=5)")
            else:
                print(f"Repeat {_r}: val_AUC {_mv:.4f}±{_sv:.4f} | test_AUC {_ma:.4f}±{_sa:.4f} (n=5)")
        except Exception as _e_rep:
            print(f"Repeat {_r} summary skipped ({_e_rep})")

if "_CV_ROWS" in globals() and len(_CV_ROWS):
    _all_val = [r[3] for r in _CV_ROWS]
    _all_auc = [r[4] for r in _CV_ROWS]
    _gv, _gsv = float(_np_cv.mean(_all_val)), float(_np_cv.std(_all_val, ddof=1)) if len(_all_val) > 1 else 0.0
    _ga, _gsa = float(_np_cv.mean(_all_auc)), float(_np_cv.std(_all_auc, ddof=1)) if len(_all_auc) > 1 else 0.0
    print(f"\nGrand (3x5={len(_CV_ROWS)} evals, BEST={_BEST}, PATIENCE={_PAT_CV}): val {_gv:.4f}±{_gsv:.4f} | test_AUC {_ga:.4f}±{_gsa:.4f}")
    if _TASK_CV in ("regression", "both"):
        _all_r2 = [r[5] for r in _CV_ROWS if _np_cv.isfinite(r[5])]
        _gr, _gsr = float(_np_cv.mean(_all_r2)), float(_np_cv.std(_all_r2, ddof=1)) if len(_all_r2) > 1 else 0.0
        print(f"Grand test_R2 {_gr:.4f}±{_gsr:.4f}")
    try:
        _orig_auc = float(res.get("test_auc", float("nan")))
        _orig_r2 = float(res.get("test_r2", float("nan"))) if "test_r2" in res else float("nan")
        _orig_val = float(res.get("best_val_r2", res.get("best_val_auc", float("nan"))))
        print(f"Original single run res: best_val={_orig_val:.4f} test_AUC={_orig_auc:.4f}" + (f" test_R2={_orig_r2:.4f}" if _np_cv.isfinite(_orig_r2) else ""))
        print(f"Comparison: grand test_AUC {_ga:.4f}±{_gsa:.4f} vs original {_orig_auc:.4f} (delta {_ga - _orig_auc:+.4f})" + (f" | grand test_R2 {_gr:.4f}±{_gsr:.4f} vs original {_orig_r2:.4f} (delta {_gr - _orig_r2:+.4f})" if _TASK_CV in ("regression", "both") and _np_cv.isfinite(_orig_r2) else ""))
    except Exception as _e_cmp:
        print(f"Comparison to res skipped ({_e_cmp})")
else:
    print("No successful CV folds.")
